# iPSCs: aggregate DVP of iPSC-derived dopaminergic neurons

Dopaminergic neurons differentiated from NGN2-inducible human iPSCs (iDA neurons), treated with α-synuclein preformed fibrils (PFF) or PBS and analysed at day 56 in vitro. Neurites and soma without aggregates, and neuritic and somatic aggregates, were cut by laser microdissection for Deep Visual Proteomics (DVP) and measured by mass spectrometry; the protein groups come from DIA-NN. This notebook makes the panels of Figure 3 and SI4.

| Short | Meaning |
|---|---|
| PFF, PBS | treated with preformed fibrils; PBS control |
| Neur, Soma | neurites, soma without aggregates |
| NAgg, SAgg | neuritic, somatic aggregates (obs values "Aggregates Neuritic", "Aggregates Somatic") |
| EA | excised aggregate (both aggregate fractions) |
| DCV | dense-core vesicle |
| log2FC | log2 fold change |
| FDR | false discovery rate, Benjamini-Hochberg |
| CV | coefficient of variation |
| PCA | principal component analysis |
| GSEA, NES | gene set enrichment analysis, normalised enrichment score |

**Inputs:** `input_dir` (DIA-NN `report.pg_matrix.tsv` and `report.pr_matrix.tsv`, `metadata.csv`), the UniProt-to-gene table at `var_metadata_path`, the MSigDB gene-set cache `src/gsea_genesets_cache.parquet` (Reactome and GO biological process), the gene list of every GSEA row label `src/gsea_label_genes.yaml`, and the GSEA decision files `review.csv` in `gsea_dir/<contrast>/`. The paths are set in the first cell of section 2.

**Outputs:** result tables in `output_dir`, GSEA results in `gsea_dir`; figure panels in `figures/`, written by `save_panel` and assembled in section 14.

# 1. Imports

In [ ]:
# --- standard library ---
import dataclasses
import glob
import logging
import os
import re
import sys
import warnings
from datetime import date

# --- scientific stack ---
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc

# --- plotting ---
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.transforms import ScaledTranslation, blended_transform_factory
from adjustText import adjust_text

# --- alphapepttools ---
import alphapepttools as at
from alphapepttools.pl.figure import create_figure, label_axes

# --- project modules in src/ ---
# The path is relative to this notebook's folder, which VS Code uses as the
# working directory of a notebook. Importing figure_style sets the matplotlib and
# alphapepttools style of every panel.
sys.path.insert(0, "src")
from figure_style import (ANNOT_FS, AXES_LW, BG_COLOR, DOWN_COLOR, DRAW_SCALE,
                          HIGHLIGHT_COLOR, LABEL_FS, LINE_LW, MARKER_LW, REF_COLOR,
                          THIN_LW, TICK_FS, TITLE_FS, UP_COLOR,
                          compose_figure, figsize, native_pt, save_panel, shade, style_summary)
from gsea_pipeline import (GSEAConfig, load_gene_sets, run_contrast, build_review,
                           assert_pass2_selection, load_label_genes, report_review_coverage,
                           report_row_redundancy, report_offlist_cells, plot_combined)
from dotplot import tested_dotplot
from heatmap import group_heatmap
from preprocessing import flag_contaminants

logging.getLogger("fontTools.subset").setLevel(logging.WARNING)
today = date.today().strftime("%Y%m%d")      # date stamp of the output file names

# 2. Data loading

In [ ]:
# Every path of the notebook, relative to the repository root.
output_dir = 'results/iDA'        # result tables
gsea_dir   = 'gsea/iDA'   # GSEA results; <contrast>/review.csv are inputs
input_dir  = 'data/iDA/DIANN_OptimizedSettings'    # DIA-NN output, metadata.csv
var_metadata_path = 'src/Uniprot_to_GeneSymbol.csv'   # alphapepttools documentation dataset
# Panel folders: save_panel writes into them, section 14 assembles them.
FIG3_DIR = "figures/Figure3_Ipsc"
SI4_DIR  = "figures/SIs/SI4_IPSCs"
os.makedirs(output_dir, exist_ok=True)

# DIA-NN protein group matrix, sample metadata and the UniProt-to-gene table.
# adata_raw is not changed after this cell; the preprocessing cell starts from a
# copy of it, so a re-run of this cell is picked up there.
adata_raw    = at.io.read_pg_table(f"{input_dir}/report.pg_matrix.tsv", search_engine="diann")
obs_metadata = pd.read_csv(f"{input_dir}/metadata.csv", index_col=0)
var_metadata = pd.read_csv(var_metadata_path, index_col=0)

# Sample metadata onto obs. Samples without a metadata row are dropped.
_n = adata_raw.n_obs
adata_raw = at.pp.add_metadata(adata=adata_raw, incoming_metadata=obs_metadata,
                               axis=0, keep_data_shape=False)
print(f"{adata_raw.n_obs} of {_n} samples have a metadata row; {_n - adata_raw.n_obs} dropped")

# Protein.Group, Protein.Names and Genes of the UniProt-to-gene table onto var,
# matched on the protein group ID. The gene symbols of the analysis come from the
# DIA-NN column genes (preprocessing cell).
adata_raw.var = adata_raw.var.merge(var_metadata, left_index=True, right_index=True, how="left")

display(adata_raw)
display(adata_raw.obs.head())
display(adata_raw.var.head())

## 2.1 Style, colours and condition order

In [ ]:
# Page grid, fonts, line widths and colour roles come from src/figure_style.py
# (imported in the first cell), shared by all notebooks of the project; printed
# here as a record of the printed sizes.
print(style_summary())

# ---------------------------------------------------------------- CONDITIONS
# Plotting order: PBS -> PFF without aggregates -> aggregates, neurites before
# soma within each.
biological_order = [
    "iDA D56 PBS N1 Neurites Without Aggregates",
    "iDA D56 PBS N1 Soma Without Aggregates",
    "iDA D56 PFF N2 Neurites Without Aggregates",
    "iDA D56 PFF N2 Soma Without Aggregates",
    "iDA D56 PFF N2 Aggregates Neuritic",
    "iDA D56 PFF N2 Aggregates Somatic",
]

# Hue = aggregate status, with the hues of the 1_HEK notebook (HEK293):
# violet = excised aggregate (EA), blue = without aggregates, grey = PBS. The
# compartment is the lightness: neurites are the lighter shade of each hue.
CATEGORY_COLORS = {
    "iDA D56 PFF N2 Aggregates Somatic":           "#8e6ba8",  # EA, soma
    "iDA D56 PFF N2 Aggregates Neuritic":          "#b79fcb",  # EA, neurites
    "iDA D56 PFF N2 Soma Without Aggregates":      "#7fb3d8",  # without aggregates, soma
    "iDA D56 PFF N2 Neurites Without Aggregates":  "#aacce6",  # without aggregates, neurites
    "iDA D56 PBS N1 Soma Without Aggregates":      "#999999",  # PBS, soma
    "iDA D56 PBS N1 Neurites Without Aggregates":  "#bfbfbf",  # PBS, neurites
}
CATEGORY_SHORT = {
    "iDA D56 PFF N2 Aggregates Somatic":           "Somatic Aggregate",
    "iDA D56 PFF N2 Aggregates Neuritic":          "Neuritic Aggregate",
    "iDA D56 PFF N2 Soma Without Aggregates":      "Soma",
    "iDA D56 PFF N2 Neurites Without Aggregates":  "Neurite",
    "iDA D56 PBS N1 Soma Without Aggregates":      "PBS Soma",
    "iDA D56 PBS N1 Neurites Without Aggregates":  "PBS Neurite",
}
assert set(biological_order) == set(CATEGORY_COLORS) == set(CATEGORY_SHORT), (
    "biological_order, CATEGORY_COLORS and CATEGORY_SHORT name different conditions")

# Built in biological_order, so cells that take their order from this dict and
# cells that take it from biological_order plot the same left-to-right sequence.
biological_color_dict = {c: CATEGORY_COLORS[c] for c in biological_order}

## 2.2 Preprocessing

In [ ]:
# ============================================================
# Preprocessing: every AnnData object the analysis uses, in one cell
# ============================================================
# Run the import, data loading and style cells and this one to start at any plot.
# Each run starts from a copy of the loaded matrix, so the cell builds the same
# objects however often it runs.
adata = adata_raw.copy()

# --- Variables ---
biological = "class_clean"
MIN_PROTEINS = 2000

# --- QC metrics: protein groups and precursors per sample (SI4A) ---
adata.obs["num_proteins"]         = np.sum(~np.isnan(adata.X), axis=1)

pr_table_path = f'{input_dir}/report.pr_matrix.tsv'
pr_matrix = pd.read_csv(pr_table_path, sep="\t", index_col=0).T
pr_matrix = pr_matrix.reindex(adata.obs.index)
pr_numeric = pr_matrix.apply(pd.to_numeric, errors="coerce")
adata.obs["num_precursors"] = np.sum(~np.isnan(pr_numeric.values), axis=1)
del pr_matrix, pr_numeric

# --- Every condition in the data has a colour and a plotting position ---
_missing_cats = set(adata.obs[biological].astype(str)) - set(biological_order)
assert not _missing_cats, (
    f"conditions present in adata.obs['{biological}'] but absent from "
    f"biological_order: {sorted(_missing_cats)}"
)

# --- Gene names and contaminant flags (src/preprocessing.py) ---
# Keratins, cRAP-only protein groups and immunoglobulins are flagged in every
# notebook; extra_contaminants holds the serum, skin, secretion and coating
# proteins of this culture (UniProt entry names or gene symbols). The flags are
# computed once here, so every object below carries them. Flagged proteins are
# removed from the t-test, PCA, imputed and targeted-panel inputs; adata_sf and
# adata keep them.
adata.var["single_gene_name"] = adata.var["genes"].str.split(";").str[0].copy()
extra_contaminants = [
    'K1C13_HUMAN', 'K1C18_HUMAN', 'K2C1B_HUMAN', 'K2C3_HUMAN',
    'K2C4_HUMAN', 'K2C72_HUMAN', 'K2C78_HUMAN', 'K2C79_HUMAN', 'K2C8_HUMAN',
    'HORN_HUMAN', 'FILA_HUMAN', 'FILA2_HUMAN', 'DCD_HUMAN', 'CDSN_HUMAN',
    'DSG1_HUMAN', 'DSC1_HUMAN', 'DSC3_HUMAN', 'DESP_HUMAN', 'PLAK_HUMAN',
    'TGM1_HUMAN', 'TGM3_HUMAN',
    'DMKN_HUMAN', 'CYTM_HUMAN', 'CYTA_HUMAN', 'S10AB_HUMAN',
    'PROL4_HUMAN',
    'ALBU_HUMAN', 'HBA_HUMAN', 'HBB_HUMAN', 'HBD_HUMAN',
    'TRFE_HUMAN', 'TRFL_HUMAN', 'LYSC_HUMAN', 'APOA1_HUMAN', 'ANT3_HUMAN',
    'IGHA1_HUMAN', 'IGHG1_HUMAN',
    'cRAP-TRYP_PIG', 'cRAP-K2M3_SHEEP',
    'INS_HUMAN', 'TRY3_HUMAN',
    'AZGP1', 'S100A7', 'BPIFA1',   # gene symbols of the next three entries
    'ZA2G_HUMAN',     # AZGP1 — zinc-α-2-glycoprotein (serum/saliva)
    'S10A7_HUMAN',    # S100A7 / psoriasin — skin
    'BPIA1_HUMAN',    # BPIFA1 / SPLUNC1 — secretory
    'CILP2_HUMAN',    # CILP2 — cartilage, probably from the matrigel coating
]
adata.var = flag_contaminants(adata.var, extra=extra_contaminants)

# --- Sample filtering ---
adata.obs["is_outlier"] = adata.obs["num_proteins"] < MIN_PROTEINS
adata_unfiltered = adata.copy()
adata = adata_unfiltered[~adata_unfiltered.obs["is_outlier"], :].copy()
adata_sf = adata.copy()

# Rank of every protein by its median intensity (Figure 3A)
adata_sf.var["median_intensity"] = np.nanmedian(adata_sf.X, axis=0)
adata_sf.var["protein_rank"]     = np.argsort(np.argsort(-adata_sf.var["median_intensity"]))

# --- Completeness filter ---
# adata: no missing value in at least one condition. adata_strict: no missing
# value in any sample.
# filter_data_completeness with group_column drops a protein that fails in any
# condition, so each condition is flagged on its own and a protein is kept if it
# passes in at least one.
def complete_in_any_condition(adata, max_missing):
    passes = np.zeros(adata.n_vars, dtype=bool)
    for condition in adata.obs[biological].unique():
        flagged = at.pp.filter_data_completeness(
            adata.copy(), max_missing=max_missing,
            group_column=biological, groups=[condition], action="flag",
        )
        passes |= flagged.var["passed_threshold_missing_values"].to_numpy()
    return passes

adata = adata[:, complete_in_any_condition(adata, max_missing=0)].copy()

adata_strict = adata[:, at.pp.filter_data_completeness(
    adata, max_missing=0, action="flag"
).var["passed_threshold_missing_values"]].copy()

# --- CV per condition, on linear intensities (SI4B) ---
# Sample SD (ddof=1); a protein with fewer than three values in a condition gets
# no CV there.
for condition in adata.obs[biological].unique():
    subset = adata[adata.obs[biological] == condition, :]
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=RuntimeWarning)
        mean_vals = np.nanmean(subset.X, axis=0)
        std_vals  = np.nanstd(subset.X, axis=0, ddof=1)
        cvs       = std_vals / mean_vals
        cvs[np.sum(~np.isnan(subset.X), axis=0) < 3] = np.nan
    adata.var[f"CV_{condition}"] = cvs

# --- Log2 copy, log2(x + 1), missing values kept ---
adata_log = adata.copy()
sc.pp.log1p(adata_log, base=2)

# --- Imputed copy: 80% completeness in at least one condition + global Perseus-style imputation ---
adata_imputed = adata_sf[:, complete_in_any_condition(adata_sf, max_missing=0.2)].copy()
sc.pp.log1p(adata_imputed, base=2)
at.pp.impute_gaussian(
    adata_imputed,
    std_offset=1.8,
    std_factor=0.3,
    random_state=42,
)

# --- PCA on the proteins complete in every sample, without contaminants ---
adata = at.pp.filter_data_completeness(adata, max_missing=0, action="flag", var_colname="is_core")
adata.var["prot_for_pca"] = adata.var["is_core"] & ~adata.var["is_contaminant"]
adata_pca = adata.copy()
sc.pp.log1p(adata_pca, base=2)
at.tl.pca(adata_pca, meta_data_mask_column_name="prot_for_pca")

# --- t-test input: complete in every sample, without contaminants, log2 ---
adata_strict_no_contaminants = adata_strict[:, ~adata_strict.var["is_contaminant"]].copy()
sc.pp.log1p(adata_strict_no_contaminants, base=2)

# adata_sf must still be the raw, sample-filtered matrix at this point: NaNs
# present and intensities on the linear scale.
assert np.isnan(adata_sf.X).any(), "adata_sf has no missing values — it was imputed in place"
assert np.nanmax(adata_sf.X) > 1e3, "adata_sf looks log-transformed"

print(f"Bootstrap complete: {adata.n_obs} samples, {adata.n_vars} proteins")
print(f"  adata_raw:                    {adata_raw.n_vars} proteins (pristine input)")
print(f"  adata_sf:                     {adata_sf.n_vars} proteins (all, sample-filtered)")
print(f"  contaminants flagged:         {int(adata_sf.var['is_contaminant'].sum())} of {adata_sf.n_vars}")
print(f"  adata_log:                    {adata_log.n_vars} proteins (log2)")
print(f"  adata_pca:                    {adata_pca.var['prot_for_pca'].sum()} core proteins for PCA")
print(f"  adata_imputed:                {adata_imputed.n_vars} proteins (80% per-cond, log2, imputed)")
print(f"  adata_strict_no_contaminants: {adata_strict_no_contaminants.n_vars} proteins (t-test input)")
print(f"  plot order:                   {' -> '.join(CATEGORY_SHORT[c] for c in biological_color_dict)}")
# Quoted in the text: six samples per condition, three somatic aggregate samples.
_n_cond = adata.obs[biological].value_counts()
print("CHECK samples per condition: "
      + ", ".join(f"{CATEGORY_SHORT[c]} {_n_cond.get(c, 0)}" for c in biological_order))

# 3. Protein groups and precursors per sample (SI4A)

In [ ]:
# SI4A, 2 x 1 units: protein groups (left axis) and precursors (right axis,
# hatched) per sample and condition, with the condition means above the boxes.
fig, axm = create_figure(1, 1, figsize=figsize(2, 1))
ax_left  = axm.next()
ax_right = ax_left.twinx()

# twinx() follows rcParams and hides the right spine, which carries the
# precursor scale here; its left spine would be drawn over ax_left's.
ax_right.spines["right"].set_visible(True)
ax_right.spines["left"].set_visible(False)
ax_left.grid(axis="x", visible=False)        # no vertical grid lines between the boxes

conditions      = biological_order
n_cond          = len(conditions)
positions_left  = np.arange(n_cond) * 2          # protein groups
positions_right = np.arange(n_cond) * 2 + 0.8    # precursors

pg_data = [adata.obs.loc[adata.obs[biological] == c, "num_proteins"].to_numpy() for c in conditions]
pr_data = [adata.obs.loc[adata.obs[biological] == c, "num_precursors"].to_numpy() for c in conditions]

S_DOT  = native_pt(2.2) ** 2     # sample dots, 2.2 pt across in print
JITTER = 0.15                    # largest horizontal offset of a dot, in x units

for ax, data, pos, hatch, marker, seed in (
        (ax_left,  pg_data, positions_left,  None,  "o", 42),
        (ax_right, pr_data, positions_right, "///", "D", 43)):
    bp = ax.boxplot(data, positions=pos, widths=0.6, patch_artist=True,
                    showfliers=False, zorder=2)
    for i, (box, cond) in enumerate(zip(bp["boxes"], conditions)):
        color = biological_color_dict[cond]
        box.set_facecolor(color)
        box.set_edgecolor("black")
        if hatch:
            box.set_hatch(hatch)
        jitter = np.random.default_rng(seed).uniform(-JITTER, JITTER, len(data[i]))
        ax.scatter(pos[i] + jitter, data[i], s=S_DOT, color=color, marker=marker,
                   edgecolors="black", linewidths=MARKER_LW, zorder=3)

# Both axes start at 0. The precursor axis is scaled by the ratio of the overall
# medians, so the two series are drawn at matching heights.
HEADROOM = 1.1
pg_max = max(d.max() for d in pg_data)
pr_max = max(d.max() for d in pr_data)
ratio  = np.median(np.concatenate(pr_data)) / np.median(np.concatenate(pg_data))
ax_left.set_ylim(0, pg_max * HEADROOM)
ax_right.set_ylim(0, max(pg_max * HEADROOM * ratio, pr_max * HEADROOM))

X_PAD = 0.7
ax_left.set_xlim(positions_left[0] - X_PAD, positions_right[-1] + X_PAD)

# Means above the boxes: protein groups in the lower row (bold), precursors in
# the upper row (bold italic). The offsets are in points, so the two rows keep
# their spacing whatever height the axes get.
GAP = native_pt(1.5)           # axes top to the lower row
ROW = 1.25 * ANNOT_FS          # lower row to upper row
top_left  = blended_transform_factory(ax_left.transData,  ax_left.transAxes)
top_right = blended_transform_factory(ax_right.transData, ax_right.transAxes)
for i in range(n_cond):
    ax_left.annotate(f"{pg_data[i].mean():.0f}", (positions_left[i], 1), xycoords=top_left,
                     xytext=(0, GAP), textcoords="offset points",
                     ha="center", va="bottom", fontsize=ANNOT_FS, fontweight="bold")
    ax_right.annotate(f"{pr_data[i].mean():.0f}", (positions_right[i], 1), xycoords=top_right,
                      xytext=(0, GAP + ROW), textcoords="offset points",
                      ha="center", va="bottom", fontsize=ANNOT_FS, fontweight="bold",
                      style="italic")

ax_left.set_xticks(positions_left + 0.4)
ax_left.set_xticklabels(["\n".join(CATEGORY_SHORT[c].rsplit(" ", 1)) for c in conditions])
ax_left.set_ylabel("Protein groups")
ax_right.set_ylabel("Precursors")

ax_left.legend(handles=[
    Patch(facecolor=BG_COLOR, edgecolor="black", linewidth=MARKER_LW, label="Protein groups"),
    Patch(facecolor=BG_COLOR, edgecolor="black", linewidth=MARKER_LW, hatch="///", label="Precursors"),
], loc="lower right")

fig.canvas.draw()
assert ax_right.spines["right"].get_visible(), "the precursor axis lost its spine"
assert ax_right.get_yticklabels(), "the precursor axis has no tick labels"

save_panel(fig, "a", "boxplot_pg_precursors", SI4_DIR)

# 4. Abundance rank plot (Figure 3A)

In [ ]:
# Figure 3A, 1 x 1 unit: every protein of adata_sf ranked by its median intensity
# across all samples, with landmark proteins called out.
fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()

S_BG = native_pt(0.8) ** 2      # background dots, 0.8 pt across in print
S_HL = native_pt(2.4) ** 2      # called-out proteins, 2.4 pt across

at.pl.scatter(
    ax=ax, data=adata_sf.var, x_column="protein_rank", y_column="median_intensity",
    color=BG_COLOR, scatter_kwargs={"s": S_BG, "linewidths": 0},
)
ax.set_yscale("log")
# The exponent in a 10^n tick label is set at 70 % size and prints below 5 pt,
# so the ticks show the exponent itself and the axis label says log10.
ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{np.log10(v):.0f}"))

# Called-out proteins, from high to low abundance
highlight_genes = [
    'TUBB3',   # neuronal tubulin: neuronal identity
    'SNCA',    # α-synuclein, the protein of the fibrils
    'VDAC1',   # mitochondrial outer membrane
    'SYT1',    # synaptic vesicle
    'MAPT',    # tau, Parkinson's disease related
    'UBQLN2',  # largest enrichment in the neuritic aggregates (Figure 3F)
    'CHGB',    # dense-core vesicle cargo
]

# Match on the split gene list, not the whole string: var["genes"] is
# semicolon-separated for merged protein groups (e.g. "HSPA1A;HSPA1B"), so
# .isin() on the raw string drops any symbol that is not alone in its group.
_rows, _hl_labels = [], []
for _idx, _raw in adata_sf.var["genes"].fillna("").items():
    _syms = [s.strip() for s in _raw.split(";")]
    _hit  = next((g for g in highlight_genes if g in _syms), None)
    if _hit is not None:
        _rows.append(_idx)
        _hl_labels.append(_hit)

highlight = adata_sf.var.loc[_rows].copy()
highlight["highlight_label"] = _hl_labels   # the wanted symbol, not the group string
print("not found:", [g for g in highlight_genes if g not in set(_hl_labels)])

ax.scatter(highlight["protein_rank"], highlight["median_intensity"], s=S_HL,
           color=HIGHLIGHT_COLOR, edgecolors="black", linewidths=MARKER_LW, zorder=5)

texts = [ax.text(row["protein_rank"], row["median_intensity"], row["highlight_label"],
                 fontsize=ANNOT_FS, ha="left", va="center")
         for _, row in highlight.iterrows()]

# Every 5th point of the curve is passed as an obstacle, so adjust_text moves
# the labels off the curve.
adjust_text(
    texts,
    x=adata_sf.var["protein_rank"].values[::5],
    y=adata_sf.var["median_intensity"].values[::5],
    ax=ax,
    arrowprops=dict(arrowstyle="-", color=REF_COLOR, lw=THIN_LW),
    expand=(1.5, 1.0),         # margin kept around points and labels, x and y, as a factor of the label size
    force_static=(1.0, 1.5),   # repulsion from the points
    force_text=(0.5, 1.0),     # repulsion between labels
    force_explode=(0.5, 0.8),  # first displacement of overlapping labels
)

label_axes(ax, xlabel="Protein rank", ylabel="log10 median intensity")

save_panel(fig, "a", "rank_abundance", FIG3_DIR)

# 5. Coefficient of variation per condition (SI4B)

In [ ]:
# SI4B, 1 x 1 unit: CV of every protein within each condition, with the mean CV
# per condition above the violins. adata (complete in at least one condition,
# linear intensities); the CVs are computed in the preprocessing cell, for every
# condition with at least three values.
fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()
ax.grid(False)        # the 20 % reference line would be drawn on top of a grid line

cv_columns = [f"CV_{c}" for c in biological_order if f"CV_{c}" in adata.var.columns]
cv_df = adata.var[cv_columns].melt(var_name="Condition", value_name="CV")
cv_df["Condition"] = cv_df["Condition"].str.replace("CV_", "", regex=False)
cv_df = cv_df.dropna()

at.pl.violinplot(
    ax=ax, data=cv_df, grouping_column="Condition", value_column="CV",
    color_dict=biological_color_dict,
)

# Read the order back from the axes instead of assuming it. violinplot sets the
# tick labels from the groups it drew, so labels, colours and means are matched
# to the violins they belong to.
conditions = [t.get_text() for t in ax.get_xticklabels()]
assert set(conditions) == set(cv_df["Condition"]), (
    "violinplot drew a different set of groups than cv_df holds: "
    f"{sorted(set(conditions) ^ set(cv_df['Condition']))}"
)

# violinplot fills at 50 % opacity; the condition colours are shown at full
# strength, as in the protein group panel.
bodies = [c for c in ax.collections if isinstance(c, mpl.collections.PolyCollection)]
assert len(bodies) == len(conditions), f"{len(bodies)} violin bodies for {len(conditions)} conditions"
for body, cond in zip(bodies, conditions):
    body.set_facecolor(biological_color_dict[cond])

ax.set_ylim(0, cv_df["CV"].quantile(0.99) * 1.1)
ax.axhline(0.2, color=REF_COLOR, linestyle="--", linewidth=AXES_LW, zorder=0)    # CV 20 %
# Six conditions in one unit leave about 5 mm per label, so the labels are rotated.
ax.set_xticklabels(["\n".join(CATEGORY_SHORT[c].rsplit(" ", 1)) for c in conditions], rotation=90)

# Mean CV per condition, in a row above the violins as in the protein group panel.
GAP = native_pt(1.5)
top = blended_transform_factory(ax.transData, ax.transAxes)
for x, cond in zip(ax.get_xticks(), conditions):
    ax.annotate(f"{cv_df.loc[cv_df['Condition'] == cond, 'CV'].mean():.2f}", (x, 1),
                xycoords=top, xytext=(0, GAP), textcoords="offset points",
                ha="center", va="bottom", fontsize=ANNOT_FS, fontweight="bold")

label_axes(ax, ylabel="CV per protein")

save_panel(fig, "b", "cv_per_condition", SI4_DIR)

# 6. PCA (Figure 3B)

In [ ]:
# Figure 3B, 1 x 1 unit: PCA of the proteins complete in every sample, without
# contaminants (adata_pca, log2). Colour = aggregate status (grey = PBS), fill =
# compartment (filled = soma, open = neurites), one marker shape. The hues are
# the base hues of CATEGORY_COLORS (style cell); the box plots show the
# compartment by lightness, this panel by fill.
MARKER = "o"
DOT    = native_pt(2.8)     # marker diameter, 2.8 pt in print

# Positions of the two compartment labels, in points from their cluster
LABEL_DX  = {"Soma": -75, "Neurites": -30}    # negative = left
LABEL_DY  = {"Soma":  15, "Neurites": -15}    # away from the cluster, so for the
                                             # lower label a smaller value moves
                                             # it up, toward the points

FRACTION_COLORS = {
    "Aggregates":         CATEGORY_COLORS["iDA D56 PFF N2 Aggregates Somatic"],
    "Without Aggregates": CATEGORY_COLORS["iDA D56 PFF N2 Soma Without Aggregates"],
}
FRACTION_SHORT = {
    "Aggregates":         "EA",
    "Without Aggregates": "Aggregate-free",
}
PBS_GREY = CATEGORY_COLORS["iDA D56 PBS N1 Soma Without Aggregates"]

def get_fraction(condition):
    # "Soma Without Aggregates" also contains "Aggregates", so the negative
    # form has to be tested first.
    return "Without Aggregates" if "Without Aggregates" in condition else "Aggregates"

def point_style(condition):
    """Colour from aggregate status (grey if PBS); filled for soma, open for neurites."""
    color  = PBS_GREY if "PBS" in condition else FRACTION_COLORS[get_fraction(condition)]
    filled = "Neurit" not in condition        # catches "Neurites" and "Neuritic"
    return color, filled

pc1 = adata_pca.obsm["X_pca_obs"][:, 0]
pc2 = adata_pca.obsm["X_pca_obs"][:, 1]

fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()

for condition in biological_order:
    mask = adata_pca.obs[biological] == condition
    if mask.sum() == 0:
        continue
    color, filled = point_style(condition)
    ax.scatter(pc1[mask], pc2[mask], marker=MARKER, s=DOT ** 2, zorder=3,
               facecolors=color if filled else "none",
               edgecolors="black" if filled else color,
               linewidth=MARKER_LW if filled else LINE_LW,
               alpha=0.85 if filled else 1.0)

# Headroom at both ends: the two direct labels are placed above one cluster and
# below the other, and text is not autoscaled.
_y0, _y1 = ax.get_ylim()
_span = _y1 - _y0
ax.set_ylim(_y0 - 0.12 * _span, _y1 + 0.12 * _span)

# Direct-label the two compartments. Which one is on top is read off the data
# instead of assumed, so a sign flip in PC2 cannot put the label on the wrong
# side. Each label is centred on its own cluster in PC1, then shifted by
# LABEL_DX / LABEL_DY.
_neurite_mask = adata_pca.obs[biological].str.contains("Neurit").to_numpy()
_soma_mask    = ~_neurite_mask
for _m, _txt in ((_soma_mask, "Soma"), (_neurite_mask, "Neurites")):
    _upper = pc2[_m].mean() > pc2[~_m].mean()
    _dy = LABEL_DY[_txt] if _upper else -LABEL_DY[_txt]
    ax.annotate(_txt, (pc1[_m].mean(), pc2[_m].max() if _upper else pc2[_m].min()),
                xytext=(LABEL_DX[_txt], _dy), textcoords="offset points",
                ha="center", va="bottom" if _upper else "top",
                fontsize=ANNOT_FS, fontweight="bold", color="0.30")

# One legend below the axes: the three colours in the first row, the two fill
# conventions in the second. The colours are square swatches: a marker there
# would read as a statement about fill, which is what the second row carries.
_leg_order = ["Without Aggregates", "Aggregates"]
_handles = [
    Patch(facecolor=FRACTION_COLORS[f], edgecolor="black", linewidth=MARKER_LW,
          label=FRACTION_SHORT[f])
    for f in _leg_order
]
_handles.append(Patch(facecolor=PBS_GREY, edgecolor="black", linewidth=MARKER_LW,
                      label="PBS"))
_handles += [
    plt.Line2D([0], [0], marker=MARKER, color='w', markerfacecolor="0.3",
               markeredgecolor='black', markeredgewidth=MARKER_LW, markersize=DOT,
               label="Soma"),
    plt.Line2D([0], [0], marker=MARKER, color='w', markerfacecolor="none",
               markeredgecolor="0.3", markeredgewidth=LINE_LW, markersize=DOT,
               label="Neurites"),
]
# At 6 pt the legend is wider than the axes of a 1 x 1 panel, so it cannot go
# inside. matplotlib fills a legend column by column, so the handles are passed
# in column order: (colour, fill), (colour, fill), (colour).
fig.legend(handles=[_handles[i] for i in (0, 3, 1, 4, 2)], loc="outside lower center",
           ncol=3, frameon=False, borderaxespad=0,
           handlelength=1.0, handletextpad=0.4, columnspacing=0.8)

var_explained = adata_pca.uns["variance_pca_obs"]["variance_ratio"]
ax.set_xlabel(f"PC1 ({var_explained[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({var_explained[1]*100:.1f}%)")

save_panel(fig, "b", "pca_biological", FIG3_DIR)

# 7. t-tests

In [ ]:
# Welch t-tests (alphapepttools diff_exp_ttest) of the seven comparisons, on the
# proteins complete in every sample, without contaminants, log2, no imputation
# (adata_strict_no_contaminants). log2fc = first group minus second group. Every
# figure panel, the GSEA (section 9.1) and the aggregate composition (section 12)
# read these results. One CSV per comparison, with gene names, in output_dir.
adata_to_use = adata_strict_no_contaminants
filter_label = "strict_no_contaminants"      # part of the CSV file names

print(f"Using {filter_label} filtering: {adata_to_use.n_vars} proteins")

# name: (group 1, group 2)
comparisons = {
    "PBS_Neurites_vs_Soma": (
        "iDA D56 PBS N1 Neurites Without Aggregates",
        "iDA D56 PBS N1 Soma Without Aggregates",
    ),
    "PFF_Neurites_vs_Soma": (
        "iDA D56 PFF N2 Neurites Without Aggregates",
        "iDA D56 PFF N2 Soma Without Aggregates",
    ),
    "Neurites_PFF_vs_PBS": (
        "iDA D56 PFF N2 Neurites Without Aggregates",
        "iDA D56 PBS N1 Neurites Without Aggregates",
    ),
    "Soma_PFF_vs_PBS": (
        "iDA D56 PFF N2 Soma Without Aggregates",
        "iDA D56 PBS N1 Soma Without Aggregates",
    ),
    "Neuritic_Agg_vs_Neurites": (
        "iDA D56 PFF N2 Aggregates Neuritic",
        "iDA D56 PFF N2 Neurites Without Aggregates",
    ),
    "Somatic_Agg_vs_Soma": (
        "iDA D56 PFF N2 Aggregates Somatic",
        "iDA D56 PFF N2 Soma Without Aggregates",
    ),
    "Neuritic_Agg_vs_Soma_Agg": (
        "iDA D56 PFF N2 Aggregates Neuritic",
        "iDA D56 PFF N2 Aggregates Somatic",
    ),
}

ttest_results = {}
for name, (group1, group2) in comparisons.items():
    result = at.tl.diff_exp_ttest(
        adata=adata_to_use,
        between_column=biological,
        comparison=(group1, group2),
    )
    if result is not None:
        ttest_results[name] = result
        n_sig = (result["fdr"] < 0.05).sum()
        n_up = ((result["fdr"] < 0.05) & (result["log2fc"] > 1)).sum()
        n_down = ((result["fdr"] < 0.05) & (result["log2fc"] < -1)).sum()
        print(f"{name}: {len(result)} tested, {n_sig} sig (FDR<0.05), {n_up} up, {n_down} down")

        # CSV with the gene name and the UniProt entry names first, sorted by log2fc
        result_export = result.merge(
            adata_to_use.var[["single_gene_name", "proteins", "uniprot_ids"]],
            left_on="protein", right_index=True, how="left",
        )
        front = ["single_gene_name", "proteins", "uniprot_ids", "protein"]
        cols = front + [c for c in result_export.columns if c not in front]
        result_export = result_export[cols].sort_values("log2fc", ascending=False)
        result_export.to_csv(f"{output_dir}/{today}_ttest_{filter_label}_{name}.csv", index=False)
    else:
        print(f"{name}: FAILED")

print(f"\nCSVs saved to {output_dir}")

## 7.1 t-tests on the imputed matrix

In [ ]:
# Welch t-tests of the seven comparisons on the imputed matrix (adata_imputed
# without contaminants: complete in at least 80 % of one condition, log2, missing
# values imputed). Read by the marker table (SI4C); the CSVs are deposited tables.
# The cell uses its own names (adata_imp_to_use, filter_label_imp,
# ttest_results_imputed), so the GSEA and the figures keep reading the t-tests on
# measured values of section 7.
adata_imp_to_use = adata_imputed[:, ~adata_imputed.var["is_contaminant"]].copy()
filter_label_imp = "imputed_no_contaminants"      # part of the CSV file names
print(f"Using {filter_label_imp} filtering: {adata_imp_to_use.n_vars} proteins")

ttest_results_imputed = {}
for name, (group1, group2) in comparisons.items():
    result = at.tl.diff_exp_ttest(
        adata=adata_imp_to_use,
        between_column=biological,
        comparison=(group1, group2),
    )
    if result is not None:
        ttest_results_imputed[name] = result
        n_sig = (result["fdr"] < 0.05).sum()
        n_up = ((result["fdr"] < 0.05) & (result["log2fc"] > 1)).sum()
        n_down = ((result["fdr"] < 0.05) & (result["log2fc"] < -1)).sum()
        print(f"{name}: {len(result)} tested, {n_sig} sig (FDR<0.05), {n_up} up, {n_down} down")

        result.to_csv(f"{output_dir}/{today}_ttest_{filter_label_imp}_{name}.csv", index=False)
    else:
        print(f"{name}: FAILED")

print(f"\nCSVs saved to {output_dir}")

# 8. Neurites against soma

## 8.1 Volcano plot, PBS neurites against soma (Figure 3C)

In [ ]:
# Compartment markers: curated nucleus-localised (soma) and process-localised
# (neurite) proteins. Figure 3C colours them, the marker table (SI4C) tests them.
soma_markers = [
    # Nuclear envelope
    'LMNA', 'LMNB1', 'LMNB2',
    # Nuclear pore (highly nucleus-restricted)
    'NUP62', 'NUP98', 'NUP153', 'RANBP2',
    # Histones (H1-2, H2AC4, H2BC11 not detected in this dataset)
    'H2AZ1', 'H1-0', 'H1-4', 'H1-5', 'H3C1',
    # RNA binding / splicing
    'HNRNPA1', 'HNRNPA2B1', 'HNRNPC', 'HNRNPD', 'HNRNPK', 'HNRNPU',
    'RBMX', 'EWSR1', 'FUS', 'ILF3', 'SFPQ', 'NONO',
    'SF3B1', 'SRSF1', 'SRSF3', 'SRSF7', 'U2AF2',
    # Nuclear other
    'NCL', 'NPM1', 'PARP1', 'TOP2B', 'XRCC5', 'XRCC6',
    'POLR2A', 'GTF2I',
]

neurite_markers = [
    # Synaptic vesicle / SNARE
    'SYT1', 'SYP', 'SYN1', 'SYN2', 'SNAP25', 'VAMP2', 'SV2A',
    'STXBP1', 'STX1A', 'STX1B', 'SYT11',
    # Pre/post-synaptic scaffolds (SHANK3 not detected)
    'BSN', 'PCLO',                  # presynaptic active zone
    'DLG4', 'HOMER1',               # postsynaptic density
    # Neuronal cytoskeleton (dendritic + axonal mix — broadly "process-localized")
    'MAP2', 'DPYSL2', 'DPYSL3', 'DPYSL5',
    'STMN1', 'STMN2', 'GAP43',
    # Axonal cytoskeleton (axon-enriched)
    'NEFL', 'NEFM', 'NEFH', 'INA',
    'MAPT',                          # tau — axonal counterpart to dendritic MAP2
    # Axonal transport
    'KIF5A', 'KIF5B', 'KIF5C', 'DYNC1H1',
    # Axonal transport accessory
    'KLC1', 'KLC2', 'DCTN1',
    # Cell adhesion / membrane
    'NCAM1', 'L1CAM', 'NRCAM', 'THY1',
    # Axonal cell adhesion
    'CNTN1', 'NFASC',
    # Ion transport
    'ATP1A1', 'ATP1A3',
    # Vesicle trafficking
    'RAB3A', 'RAB3C',
]

In [ ]:
# Figure 3C, 1 x 1 unit: volcano plot, PBS neurites against PBS soma (t-test of
# section 7), with the compartment markers coloured and a few of them labelled.
name = "PBS_Neurites_vs_Soma"

always_label = ["SNCA"]
LFC_THRESH   = 0.5

# marker diameters in print
DOT_BG, DOT_MARKER, DOT_SNCA = native_pt(1.0), native_pt(1.8), native_pt(2.4)

result = ttest_results[name].copy()
result = result.merge(adata_log.var[["single_gene_name"]], left_on="protein", right_index=True, how="left")
result["status"] = "unchanged"
result.loc[(result["fdr"] < 0.05) & (result["log2fc"] >  LFC_THRESH), "status"] = "upregulated"
result.loc[(result["fdr"] < 0.05) & (result["log2fc"] < -LFC_THRESH), "status"] = "downregulated"

# Layer of each protein: SNCA, a marker set, or else its direction of change
result["plot_layer"] = result["status"]
is_neurite = result["single_gene_name"].isin(neurite_markers)
is_soma    = result["single_gene_name"].isin(soma_markers)
is_snca    = result["single_gene_name"].isin(always_label)
result.loc[is_neurite, "plot_layer"] = "neurite"
result.loc[is_soma,    "plot_layer"] = "soma"
result.loc[is_snca,    "plot_layer"] = "snca"

# Colours defined once, so the layers and the legend use the same values.
# Compartment is a fourth axis, orthogonal to fraction, direction and the
# call-out, so it needs hues none of those own. Sand and indigo lie on the
# blue-yellow axis, the one red-green deficiency preserves: under a Vienot
# dichromat simulation they separate at 0.63, green and orange at 0.14.
NEURITE_C, SOMA_C = "#DDCC77", "#332288"
VOLCANO_COLORS = {
    "neurite": NEURITE_C,
    "soma":    SOMA_C,
    "snca":    HIGHLIGHT_COLOR,
    "up_bg":   shade(UP_COLOR,   0.35),
    "down_bg": shade(DOWN_COLOR, 0.35),
    "ns":      BG_COLOR,
}

fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()
ax.spines[["top", "right"]].set_visible(True)     # boxed, as the Figure 2 volcano

# Marker settings per layer; volcano() passes the "scatter_kwargs" of a layer to scatter().
_marker = {"s": DOT_MARKER ** 2, "alpha": 1.0, "zorder": 5, "edgecolor": "black", "linewidth": MARKER_LW}
at.pl.volcano(
    data=result, x_column="log2fc", y_column="-log10(fdr)", ax=ax,
    x_thresholds=(-LFC_THRESH, LFC_THRESH),
    layers=[
        ("plot_layer", "neurite",       "neurite", {"scatter_kwargs": {**_marker}}),
        ("plot_layer", "soma",          "soma",    {"scatter_kwargs": {**_marker}}),
        ("plot_layer", "snca",          "snca",    {"scatter_kwargs": {"s": DOT_SNCA ** 2, "alpha": 1.0, "zorder": 6,
                                                                       "marker": "D", "edgecolor": "black",
                                                                       "linewidth": MARKER_LW}}),
        ("plot_layer", "upregulated",   "up_bg",   {"scatter_kwargs": {"s": DOT_BG ** 2, "alpha": 0.6, "linewidth": 0}}),
        ("plot_layer", "downregulated", "down_bg", {"scatter_kwargs": {"s": DOT_BG ** 2, "alpha": 0.6, "linewidth": 0}}),
        ("plot_layer", "unchanged",     "ns",      {"scatter_kwargs": {"s": DOT_BG ** 2, "alpha": 0.5, "linewidth": 0}}),
    ],
    color_dict=VOLCANO_COLORS,
    label_layers=None, display_id_column="single_gene_name",
    scatter_kwargs={"s": DOT_BG ** 2}, legend=None,
)
ax.set_title("")   # at.pl.volcano sets the title "Volcano Plot" and has no parameter for it

# The only lines on the axes so far are the threshold lines, which the volcano
# function draws 1 pt wide on the canvas (0.22 pt in print, below the minimum).
for _line in ax.lines:
    _line.set(linewidth=THIN_LW, color=REF_COLOR)

# Proteins labelled with their name; every marker is coloured through the
# volcano layers above.
text_label_genes = [
    "SNCA",
    # Soma
    "H1-0",      # histone (largest effect)
    "FUS",       # splicing, ALS-related
    # Neurite
    "SYP",       # synaptic vesicle (synaptophysin)
    "SNAP25",    # SNARE complex
    "MAPT",      # axonal cytoskeleton (tau)
]

label_set = result[result["single_gene_name"].isin(text_label_genes)]

# Without NaN and infinite coordinates, which stop adjust_text with an error
finite_mask = (
    np.isfinite(label_set["log2fc"].values)
    & np.isfinite(label_set["-log10(fdr)"].values)
)
label_set = label_set[finite_mask]

# Labels in a white box, so they stay readable where they are placed over a dot.
texts = []
for _, row in label_set.iterrows():
    t = ax.text(
        row["log2fc"], row["-log10(fdr)"], row["single_gene_name"],
        fontsize=ANNOT_FS, zorder=10,
        bbox=dict(boxstyle="round,pad=0.1", facecolor="white",
                  edgecolor="none", alpha=0.85),
    )
    texts.append(t)

# Force a layout pass so text bboxes are computable before adjust_text reads them.
fig.canvas.draw()

# Every dot is passed as an obstacle, so adjust_text moves the labels off the
# dots and into empty space, with a leader line to their dot. expand is the
# margin kept around each label and dot, as a factor of the label size.
_pts = result[np.isfinite(result["log2fc"]) & np.isfinite(result["-log10(fdr)"])]
adjust_text(
    texts,
    x=_pts["log2fc"].values, y=_pts["-log10(fdr)"].values,
    ax=ax,
    arrowprops=dict(arrowstyle="-", color=REF_COLOR, lw=THIN_LW),
    force_text=(0.6, 0.8),
    force_static=(0.6, 0.8),
    expand=(1.6, 1.6),
)

# Legend — colours read from VOLCANO_COLORS. SNCA is omitted: its diamond is
# already labelled on the plot.
_entry = dict(marker="o", ls="none", ms=DOT_MARKER, mew=MARKER_LW)
legend_handles = [
    Line2D([], [], mfc=VOLCANO_COLORS["neurite"], mec="black", label="Neurite markers", **_entry),
    Line2D([], [], mfc=VOLCANO_COLORS["soma"],    mec="black", label="Soma markers", **_entry),
    Line2D([], [], mfc=VOLCANO_COLORS["up_bg"],   mec="none",  label="Upregulated", **_entry),
    Line2D([], [], mfc=VOLCANO_COLORS["down_bg"], mec="none",  label="Downregulated", **_entry),
    Line2D([], [], mfc=VOLCANO_COLORS["ns"],      mec="none",  label="Not significant", **_entry),
]
# Below the axes in two columns, one pair per row: the two marker sets, up and
# down, then not significant. matplotlib fills a legend column by column, so the
# handles are passed in column order.
fig.legend(handles=[legend_handles[i] for i in (0, 2, 4, 1, 3)], loc="outside lower center",
           ncol=2, frameon=False, borderaxespad=0, borderpad=0,
           labelspacing=0.3, handlelength=1.0, handletextpad=0.4, columnspacing=1.0)
# h_pad is the constrained-layout padding in inches; three of them are between
# the x axis label and the legend.
fig.get_layout_engine().set(h_pad=0.01)

label_axes(ax, xlabel="log2FC (Neurite / Soma)", ylabel="−log10 FDR")

save_panel(fig, "c", f"volcano_{name}_markers", FIG3_DIR)

## 8.2 Compartment marker table (SI4C)

In [ ]:
# SI4C, 2 x 2 units: compartment markers, PBS neurites against PBS soma (t-test
# on the imputed matrix, section 7.1). One table per expected compartment, side
# by side, because about 80 rows do not fit one column of a page. A marker
# matches when FDR < 0.05 and |log2FC| > 0.5 in its expected direction. The
# table is also written as a CSV next to the panel.
comp = "PBS_Neurites_vs_Soma"
res = ttest_results_imputed[comp].copy()
res = res.merge(adata_log.var[["single_gene_name"]], left_on="protein", right_index=True, how="left")

def observed(lfc, fdr):
    if fdr < 0.05 and lfc > 0.5:
        return "Neurite"
    if fdr < 0.05 and lfc < -0.5:
        return "Soma"
    return "n.s."

rows = []
for expected, markers in (("Soma", soma_markers), ("Neurite", neurite_markers)):
    for gene in markers:
        match = res[res["single_gene_name"] == gene]
        if not match.empty:
            r = match.iloc[0]
            rows.append({"Gene": gene, "log2FC": r["log2fc"], "FDR": r["fdr"],
                         "Expected": expected, "Observed": observed(r["log2fc"], r["fdr"])})
table_df = pd.DataFrame(rows)

# Quoted in the text: markers enriched in their expected compartment.
for expected in ("Soma", "Neurite"):
    sub = table_df[table_df["Expected"] == expected]
    print(f"CHECK {expected} markers as expected: {int((sub['Observed'] == expected).sum())} of {len(sub)}"
          f" (opposite {int(((sub['Observed'] != expected) & (sub['Observed'] != 'n.s.')).sum())})")

# --- Table panel ---
FILL = {"match": "#c8e6c9", "opposite": "#ffcdd2", "n.s.": "#fff9c4"}   # pale green / red / yellow
HEADER_FILL, EDGE = "#f0f0f0", "#cccccc"
COLS   = ["Gene", "log2FC", "FDR", "Observed"]       # Expected is the table's title
WIDTHS = [0.34, 0.18, 0.24, 0.24]                    # fractions of the table width

# Table text prints at 4.75 pt (5.0 x 0.95), below Nature's 5 pt minimum; set on
# purpose.
TABLE_FS = native_pt(5.0)*0.95

fig, axm = create_figure(1, 2, figsize=figsize(2, 2))

n_max = table_df["Expected"].value_counts().max() + 1      # rows of the longer table, header included
for expected in ("Soma", "Neurite"):
    ax = axm.next()
    ax.axis("off")
    sub = table_df[table_df["Expected"] == expected]
    text = [[g, f"{l:.2f}".replace("-", "−"), f"{f:.1e}".replace("-", "−"), o]
            for g, l, f, o in sub[COLS].itertuples(index=False)]
    fills = [[FILL["n.s." if o == "n.s." else ("match" if o == expected else "opposite")]] * len(COLS)
             for o in sub["Observed"]]
    # Both tables have the row height of the longer one, so the shorter table
    # ends higher instead of getting taller rows.
    h = (len(sub) + 1) / n_max
    tab = ax.table(cellText=text, colLabels=COLS, cellColours=fills,
                   colColours=[HEADER_FILL] * len(COLS), colWidths=WIDTHS,
                   cellLoc="center", bbox=[0, 1 - h, 1, h])
    tab.auto_set_font_size(False)
    tab.set_fontsize(TABLE_FS)
    for (r, c), cell in tab.get_celld().items():
        cell.set_edgecolor(EDGE)
        if r == 0:
            cell.set_text_props(fontweight="bold")
        # The table centres each line box, whose space for descenders puts
        # capitals and digits above the middle of the cell. The baseline is set
        # 0.36 font sizes below the cell centre instead, which centres them. The
        # table positions its text in pixels, so the shift is added to the text's
        # transform.
        _t = cell.get_text()
        _t.set_verticalalignment("baseline")
        _t.set_transform(mpl.transforms.IdentityTransform()
                         + ScaledTranslation(0, -0.36 * TABLE_FS / 72, fig.dpi_scale_trans))
    ax.set_title(f"{expected} markers")
    if expected == "Soma":
        ax_soma, h_soma = ax, h          # the shorter table; the legend goes under it

# Legend in the empty space under the soma table, which is shorter than the
# neurite table; below both tables it would take a strip of the panel height.
ax_soma.legend(handles=[Patch(facecolor=FILL[k], edgecolor=EDGE, label=lab) for k, lab in
                        (("match", "as expected"), ("opposite", "opposite"), ("n.s.", "not significant"))],
               title="FDR < 0.05, |log2FC| > 0.5", loc="upper center",
               bbox_to_anchor=(0.5, 1 - h_soma), ncol=1, frameon=False)

svg = save_panel(fig, "c", "marker_validation_table", SI4_DIR)
table_df.to_csv(svg.with_suffix(".csv"), index=False)
print(f"table: {svg.with_suffix('.csv').name}, {len(table_df)} rows")

# 9. PFF response

## 9.1 GSEA, PFF against PBS in soma and neurites (SI4D)

The decisions in `review.csv` (representative term, exclusion and display label of each high-priority cluster) were made by the authors with the help of an AI review (Claude, Anthropic) that followed the rules in `GSEA_REVIEW_PROMPT.md`; they are an input of this analysis. A re-run of [2/3] carries each decision over by term name, except for a term whose cluster changed: that term is marked MOVED and keeps no decision. `gsea_dir/REVIEW_NOTES_ipsc_contrasts.md` records the reasoning of the AI review, made for six contrasts; SI4D draws two of them.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# GSEA [1/3]: PFF against PBS, in soma and in neurites (SI4D).
# Rank : Welch t-statistic, computed by the pipeline from adata_to_use
#        (adata_strict_no_contaminants: measured values only, no imputation).
# Null : gene permutation (prerank) only. The phenotype null is off: at these
#        group sizes it cannot resolve the down direction.
# Redundant terms are clustered by leading-edge overlap and decided in a review
# file (next cell), which is replayed and asserted before anything is plotted.
# The computation is in src/gsea_pipeline.py, with the gene-set cache.
# ═══════════════════════════════════════════════════════════════════════

# Excluded on lineage or physiology grounds ONLY: each name requires a cell type,
# tissue or organ-level process absent from an iPSC-derived dopaminergic neuron
# culture, so no gene content could make the term mean what it says. Synaptic,
# neuritic and vesicle terms are deliberately absent from this list — they are
# the biology under study and are judged on their core genes in the review.
IPSC_ARTIFACT_PATHWAYS = frozenset({
    # Vascular / blood — no endothelium, no vessels, no plasma in the dish
    "GOBP_REGULATION_OF_BLOOD_CIRCULATION",
    "GOBP_REGULATION_OF_SYSTEMIC_ARTERIAL_BLOOD_PRESSURE",
    "GOBP_VASCULAR_PROCESS_IN_CIRCULATORY_SYSTEM",
    "REACTOME_CELL_SURFACE_INTERACTIONS_AT_THE_VASCULAR_WALL",
    "REACTOME_PLATELET_ACTIVATION_SIGNALING_AND_AGGREGATION",
    "REACTOME_HEMOSTASIS",
    "GOCC_BLOOD_MICROPARTICLE",

    # Immune — the named effector cell types are absent from the culture
    "REACTOME_FCGAMMA_RECEPTOR_FCGR_DEPENDENT_PHAGOCYTOSIS",
    "REACTOME_CROSS_PRESENTATION_OF_SOLUBLE_EXOGENOUS_ANTIGENS_ENDOSOMES",
    "REACTOME_ANTIGEN_PROCESSING_CROSS_PRESENTATION",
    "REACTOME_INTERFERON_GAMMA_SIGNALING",
    "REACTOME_TCR_SIGNALING",
    "REACTOME_INTERLEUKIN_1_SIGNALING",
    "REACTOME_INTERLEUKIN_1_FAMILY_SIGNALING",
    "REACTOME_CLEC7A_DECTIN_1_SIGNALING",

    # Infection — these cultures are not infected. The gene content is real
    # (translation, trafficking), which is exactly why the name misleads.
    "REACTOME_PARASITE_INFECTION",
    "REACTOME_INFLUENZA_INFECTION",
    "REACTOME_SARS_COV_INFECTIONS",
    "REACTOME_SARS_COV_1_INFECTION",
    "REACTOME_SARS_COV_2_INFECTION",
    "REACTOME_SARS_COV_1_HOST_INTERACTIONS",
    "REACTOME_SARS_COV_2_HOST_INTERACTIONS",
    "REACTOME_SARS_COV_1_MODULATES_HOST_TRANSLATION_MACHINERY",
    "REACTOME_SARS_COV_2_MODULATES_HOST_TRANSLATION_MACHINERY",
    "REACTOME_INTERACTIONS_OF_REV_WITH_HOST_CELLULAR_PROTEINS",
    "REACTOME_INTERACTIONS_OF_VPR_WITH_HOST_CELLULAR_PROTEINS",
    "REACTOME_NUCLEAR_IMPORT_OF_REV_PROTEIN",
    "REACTOME_THE_ROLE_OF_NEF_IN_HIV_1_REPLICATION_AND_DISEASE_PATHOGENESIS",

    # Embryonic and non-neural tissue programmes — the cells are post-mitotic
    # neurons, past the stage these terms describe
    "REACTOME_GASTRULATION",
    "REACTOME_SOMITOGENESIS",
    "REACTOME_FORMATION_OF_PARAXIAL_MESODERM",
    "GOBP_GASTRULATION",
    "GOBP_SOMITOGENESIS",
    "GOBP_MESODERM_DEVELOPMENT",
    "GOBP_FORMATION_OF_PRIMARY_GERM_LAYER",
    "GOBP_HEART_DEVELOPMENT",
    "GOBP_CARDIAC_MUSCLE_TISSUE_DEVELOPMENT",
    "GOBP_OSTEOBLAST_DIFFERENTIATION",
    "GOBP_EPITHELIAL_CELL_DEVELOPMENT",

    # Cochlear hair cells — absent cell type
    "REACTOME_SENSORY_PROCESSING_OF_SOUND",
})

cfg_ipsc = GSEAConfig(
    output_root=gsea_dir,
    artifact_pathways=IPSC_ARTIFACT_PATHWAYS,
    n_review_clusters=40,
    run_phenotype=False,
    # A term whose permutation null was constant and equal to the observed score
    # returns NES = 1.0 and p = 1.0; these values are not statistics, so the term
    # is dropped. run_contrast prints the number per contrast (Methods).
    sentinel_action="drop",
    # The style sizes from src/figure_style.py. plot_combined reads these for the
    # tick labels, the axis labels and the two legend keys.
    tick_fontsize=TICK_FS,
    label_fontsize=LABEL_FS,
    title_fontsize=TITLE_FS,
    cmap_colors=(DOWN_COLOR, "white", UP_COLOR),
)

gene_sets_ipsc = load_gene_sets(cfg_ipsc)   # cached in src/gsea_genesets_cache.parquet

# The two contrasts, in the order they are plotted. Each is a separate
# permutation run with seed 42. The conditions are read from `comparisons` of the
# t-test cell, so the GSEA and the t-tests compare the same samples under one name.
ipsc_contrasts = [
    "Soma_PFF_vs_PBS",
    "Neurites_PFF_vs_PBS",
]
ipsc_labels = ["Soma PFF/PBS", "Neur PFF/PBS"]

_missing = [n for n in ipsc_contrasts if n not in comparisons]
assert not _missing, f"{_missing} not in `comparisons`; run the t-test cell first"

gsea_res_ipsc = {}
for name, label in zip(ipsc_contrasts, ipsc_labels):
    cond_a, cond_b = comparisons[name]          # (numerator, denominator)
    print(f"\n{'='*78}\n{name}   {label}\n  {cond_a}\n  vs {cond_b}\n{'='*78}")
    gsea_res_ipsc[name] = run_contrast(adata_to_use, biological, cond_a, cond_b,
                                       name, gene_sets_ipsc, cfg_ipsc)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# GSEA [2/3]: one decision file per contrast. The significant terms of each
# direction are clustered on leading-edge overlap (complete linkage: every pair in
# a cluster shares at least half of the smaller leading edge, so clusters cannot
# chain), and review.csv records one representative and one label per cluster.
# build_review carries the decisions already in review.csv over, matched by term
# name; a term whose cluster changed is marked MOVED and keeps no decision. It
# drops no term.
# ═══════════════════════════════════════════════════════════════════════
for name in ipsc_contrasts:
    build_review(gsea_res_ipsc[name], cfg_ipsc)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# GSEA [3/3]: SI4D, 2 x 2 units. Rows are the labels of the decision files; one
# gene set per row: every term tested is scored against the label's gene list in
# src/gsea_label_genes.yaml, the term is chosen in the contrast where the module
# is enriched, and that term is read in both columns. A row needs at least 45 %
# of its leading edge in the list where it is chosen and 30 % in every column
# drawn filled, so dot size is comparable along a row (Methods: k = 2, rows
# selected over these two contrasts).
#   filled   FDR < 0.05 in this contrast
#   hollow   tested, FDR >= 0.05
#   grey x   outside this contrast's detected-size window, never tested
# ═══════════════════════════════════════════════════════════════════════
PFF_CONTRASTS = ipsc_contrasts
PFF_LABELS    = ipsc_labels

PFF_ROWS_UP, PFF_ROWS_DOWN = 20, 20
PFF_SIZE = (2, 2)                    # panel size in units (columns, rows)
# Dot area at the smallest and the largest detected set size, from printed
# diameters of 2.0 and 5.0 pt. The check below prints the largest dot against
# the row pitch.
PFF_DOT_AREA = (native_pt(2.0) ** 2, native_pt(5.0) ** 2)
PFF_MIN_ROWS_PER_CONTRAST = 3

# Display settings live in this cell, so changing one costs a redraw and not the
# preranks. The line widths are the style values: plot_combined's defaults print
# below 0.25 pt on this canvas.
cfg_pff = dataclasses.replace(cfg_ipsc,
                              dot_area_min=PFF_DOT_AREA[0],
                              dot_area_max=PFF_DOT_AREA[1],
                              edge_lw_sig=MARKER_LW, edge_lw_ns=AXES_LW,
                              cross_lw=AXES_LW, grid_lw=THIN_LW,
                              key_edge_lw=MARKER_LW, key_marker_size=native_pt(3.0),
                              key_mew_sig=MARKER_LW, key_mew_ns=AXES_LW,
                              keys_position="corner", contrast_label_rotation=90)

for name in PFF_CONTRASTS:
    assert_pass2_selection(pd.read_csv(cfg_pff.contrast_dir(name) / "review.csv",
                                       keep_default_na=False))

# Only a labelled cluster can become a row, so a contrast reviewed less than the
# other produces a near-empty column. The figure's own floor is 3 rows per
# contrast; 10 is what a review should reach.
report_review_coverage(PFF_CONTRASTS, cfg_pff, min_labelled=10)

# save=True also writes the row table (_grid.csv) into cfg.output_root; the
# panel itself is written by save_panel below. Two lines per contrast label:
# "Soma" above "PFF/PBS".
fig_pff, ax_pff = plot_combined(PFF_CONTRASTS, cfg_pff,
                                contrast_labels=["\n".join(l.rsplit(" ", 1)) for l in PFF_LABELS],
                                rows_up=PFF_ROWS_UP, rows_down=PFF_ROWS_DOWN,
                                figsize=figsize(*PFF_SIZE), save=True,
                                save_stem="pff_contrasts_dotplot",
                                cell_terms="purity",
                                purity_floor=0.45,
                                coverage_floor=0.10,
                                cell_purity_floor=0.30,
                                anchor_rule="coverage",
                                min_rows_per_contrast=PFF_MIN_ROWS_PER_CONTRAST)

# ----------------------------------------------------------------- CHECKS
# Printed, not asserted: the panel is saved either way, and save_panel reports
# anything drawn past the canvas edge (legends, clipped row labels).
if fig_pff is not None:
    fig_pff.canvas.draw()
    _rows = len(ax_pff.get_yticklabels())
    # Quoted in the SI4D legend: the number of rows.
    print(f"CHECK rows drawn: {_rows}")
    _ax_w = ax_pff.get_window_extent().width / fig_pff.dpi
    print(f"CHECK plotting area {_ax_w:.1f} in wide on the canvas "
          f"({_ax_w / len(PFF_CONTRASTS):.1f} in per contrast column)")
    if _ax_w < 1.5 * len(PFF_CONTRASTS):
        print("  !! the row labels and the keys leave little room for the columns")
    # Largest dot against the row pitch: dots that exceed it collide vertically.
    _pitch = ax_pff.get_window_extent().height / fig_pff.dpi * 72 / _rows
    _dmax = np.sqrt(PFF_DOT_AREA[1])
    print(f"CHECK largest dot {_dmax / DRAW_SCALE:.1f} pt across, "
          f"row pitch {_pitch / DRAW_SCALE:.1f} pt (printed)")
    if _dmax >= _pitch:
        print("  !! dots are taller than the row pitch; lower PFF_DOT_AREA[1]")

    save_panel(fig_pff, "d", "gsea_pff_contrasts", SI4_DIR)

# ----------------------------------------------------------------- REPORTS
# Advisory. Redundancy: row pairs whose terms share most of a leading edge; two
# rows that overlap in genes can be two findings, so they stay apart when they
# separate in another column. Off-list: hollow cells for which another
# significant set covers the module; each one is a question about the label's
# gene list, not about the figure.
_raws = {c: pd.read_csv(sorted(glob.glob(str(cfg_pff.contrast_dir(c) / "*_raw.csv")))[-1],
                        keep_default_na=False).set_index("term") for c in PFF_CONTRASTS}
_grid = pd.read_csv(sorted(glob.glob(
    f"{cfg_pff.output_root}/*_pff_contrasts_dotplot_grid.csv"))[-1])
report_row_redundancy(_grid, _raws, PFF_CONTRASTS, thresh=0.5)
report_offlist_cells(_grid, _raws, PFF_CONTRASTS, load_label_genes(), cfg_pff)

## 9.2 PFF response heatmap (Figure 3D)

In [ ]:
# Rows and columns of the PFF response heatmap (Figure 3D). The proteins are
# leading-edge proteins of the gene sets of SI4D that are in the strict t-test
# tables, plus five PD-linked proteins added by hand (gene: description; the
# description is for the reader and is not drawn).
heatmap_comparisons = [
    "Soma_PFF_vs_PBS",
    "Neurites_PFF_vs_PBS",
]
heatmap_labels = [
    "Soma PFF / PBS",
    "Neurites PFF / PBS",
]

# --- SNCA and PD-linked proteins (added by hand) ---
pd_markers = {
    "SNCA":  "α-synuclein",
    "SNCG":  "γ-synuclein",
    "MAPT":  "Tau",
    "PARK7": "DJ-1 (PARK7)",
    "UCHL1": "UCHL1",
}

# --- Mitochondrial OXPHOS (shared UP) ---
oxphos_markers = {
    "NDUFS1": "Complex I (NDUFS1)",
    "COX4I1": "Complex IV (COX4I1)",
    "CYCS":   "Cytochrome c",
}

# --- Cholesterol biosynthesis (shared DOWN) ---
sterol_markers = {
    "HMGCS1": "HMGCS1",
    "FDFT1":  "FDFT1 (Squalene synthase)",
    "DHCR7":  "DHCR7",
}

# --- Ribosome biogenesis (shared DOWN, stronger in neurites) ---
# NOP2 in place of WDR12, which is not in the strict tables
ribosome_biogenesis_markers = {
    "NOP2": "NOP2",
    "PES1": "Pescadillo (PES1)",
    "BOP1": "BOP1",
}

# --- Integrin-ECM adhesion (shared DOWN) ---
integrin_markers = {
    "ITGB1":  "Integrin β1",
    "ILK":    "ILK",
    "FERMT2": "Kindlin-2 (FERMT2)",
}

# --- Proteasome: 19S down in both; 20S down in soma, up in neurites ---
proteasome_markers = {
    "PSMD7": "PSMD7 (19S)",
    "PSMC2": "PSMC2 (19S)",
    "PSMA7": "PSMA7 (20S)",
    "PSMB4": "PSMB4 (20S)",
}

# --- Chaperones and ubiquitin (chaperones down and UBQLN2 up in both; UBB up in neurites) ---
proteostasis_markers = {
    "HSPH1":  "HSP110 (HSPH1)",
    "DNAJB1": "HSP40 (DNAJB1)",
    "PTGES3": "p23 (PTGES3)",
    "UBQLN2": "Ubiquilin-2",
    "UBB":    "Ubiquitin (UBB)",
}

# --- Synaptic adhesion (neurite UP) ---
# NRXN1, NLGN2 and PCDH17 are not in the strict tables
synaptic_adhesion_markers = {
    "NRCAM": "NrCAM",
    "CADM1": "SynCAM1 (CADM1)",
    "CADM3": "SynCAM3 (CADM3)",
}

# --- Synaptic vesicle cycle (neurite UP) ---
sv_cycle_markers = {
    "SYT1":   "Synaptotagmin-1",
    "CLTB":   "Clathrin light B",
    "DNM1":   "Dynamin-1",
    "DNAJC6": "Auxilin (DNAJC6)",
    "SYNJ1":  "Synaptojanin-1",
}

# --- Ribosomal proteins / translation (neurite DOWN) ---
ribosome_markers = {
    "RPS3":  "RPS3",
    "RPL11": "RPL11",
    "EEF2":  "eEF2",
}

# --- Lysosomal catabolism (soma UP) ---
# GALC, SMPD1 and IDUA are not in the strict tables
lysosomal_markers = {
    "PSAP":  "Prosaposin",
    "ASAH1": "Acid ceramidase",
    "HEXB":  "Hexosaminidase B",
    "GNS":   "GlcNAc-6-sulfatase (GNS)",
}

# --- ER-Golgi transport (soma UP) ---
er_golgi_markers = {
    "SEC23A": "SEC23A (COPII)",
    "SEC24C": "SEC24C (COPII)",
    "COG4":   "COG4",
}

# --- Rab GTPases (soma DOWN) ---
rab_markers = {
    "RAB3B":  "RAB3B",
    "RAB10":  "RAB10",
    "RAB11B": "RAB11B",
}

# Heatmap groups in row order: (label, marker dict). One line per label; two
# pairs of neighbouring groups are merged, so the labels fit the 1 x 2 panel.
all_groups = [
    ("PD-linked",    pd_markers),                     # SNCA is its first row
    ("OXPHOS",       oxphos_markers),
    ("Cholesterol",  sterol_markers),
    ("Nucleolar",    ribosome_biogenesis_markers),    # ribosome biogenesis: NOP2, PES1, BOP1
    ("Integrin-ECM", integrin_markers),
    ("Proteostasis", {**proteasome_markers, **proteostasis_markers}),    # proteasome + chaperones, ubiquitin
    ("Synapse",      {**synaptic_adhesion_markers, **sv_cycle_markers}), # synaptic adhesion + synaptic vesicles
    ("Translation",  ribosome_markers),               # ribosomal proteins: RPS3, RPL11, EEF2
    ("Lysosomal",    lysosomal_markers),
    ("ER-Golgi",     er_golgi_markers),
    ("Rab GTPases",  rab_markers),
]

In [ ]:
# Figure 3D, 1 x 2 units: log2FC of PFF against PBS in soma and in neurites
# (t-tests of section 7), for the proteins of all_groups, grouped by module.
# Drawn by group_heatmap (src/heatmap.py), which prints the number of genes drawn,
# missing cells, requested genes not shown and the row pitch.
# The colour scale ends at +-COLOR_CAP: SNCA (log2FC about 6 in soma and 8 in
# neurites) would otherwise leave every other cell near white. Cells beyond the
# cap carry their value. Gene names print at 5.0 pt: at 1 x 2 units the rows are
# 4.5 pt apart.
COLOR_CAP = 3.5

missing = [c for c in heatmap_comparisons if c not in ttest_results]
if missing:
    raise KeyError(f"Missing comparisons in ttest_results: {missing}. "
                   f"Available: {sorted(ttest_results.keys())}")

# log2FC matrix: gene symbols x comparisons
heatmap_data = {}
for comp in heatmap_comparisons:
    result = ttest_results[comp]
    result = result.merge(adata_log.var[["single_gene_name"]],
                          left_on="protein", right_index=True, how="left")
    heatmap_data[comp] = result.set_index("single_gene_name")["log2fc"]
lfc_matrix = pd.DataFrame(heatmap_data)
# Column names horizontal on three lines, "Soma" / "PFF /" / "PBS", each line
# centred: on two lines the "PFF / PBS" lines overlapped (measured -1.1 mm).
lfc_matrix.columns = ["\n".join([l.split(" ", 1)[0], *l.split(" ", 1)[1].rsplit(" ", 1)])
                      for l in heatmap_labels]

# Group names at three depths, and the first and last aligned to the panel edges:
# the blocks are 3-9 rows (4.7-14 mm) and the names up to 14 mm long.
group_heatmap(lfc_matrix, all_groups, "d", "heatmap_pff_stress_modules", FIG3_DIR,
              size=(1, 2), norm=TwoSlopeNorm(vmin=-COLOR_CAP, vcenter=0, vmax=COLOR_CAP),
              gene_fs=native_pt(5.0), label_gap_pt=3.0, stagger=3, align_ends=True,
              label_above=COLOR_CAP, col_linespacing=1.0)

# Quoted in the text: the α-synuclein fold change in soma and in neurites.
for comp in heatmap_comparisons:
    _v = heatmap_data[comp]["SNCA"]
    print(f"CHECK SNCA {comp}: log2FC {_v:+.2f}, {2 ** _v:.0f}-fold")

# 10. Dense-core vesicle proteins

In [ ]:
# Dense-core vesicle (DCV) proteins, curated: the cargo, made in the soma and
# carried to the neurites, and the release machinery at the neurite terminals.
# Figure 3E colours them, SI4E shows them across four comparisons.
dcv_cargo = [
    'CHGA', 'CHGB', 'SCG2', 'SCG3', 'SCG5',   # granins
    'VGF', 'PCSK1N',                          # neuropeptides
    'CPE', 'PAM', 'PCSK1', 'PCSK2',           # prohormone processing
]
dcv_machinery = [
    'RAB3A', 'RAB3B', 'RAB3C', 'RAB3D',       # Rab3
    'RAB27A', 'RAB27B',                       # Rab27, DCV-specific
    'SV2A', 'SV2B', 'SV2C',                   # SV2
    'SYT1', 'SYT4', 'SYT11',                  # synaptotagmins
    'CADPS', 'CADPS2',                        # priming
    'VAMP4', 'VAMP2',                         # v-SNAREs
    'HID1',                                   # biogenesis
]

## 10.1 Neurite-to-soma ratio, PBS against PFF (Figure 3E)

In [ ]:
# Figure 3E, 1 x 1 unit: neurite / soma log2FC in PBS (x) against PFF (y) for
# every protein (t-tests of section 7), with the DCV proteins coloured.
name_pbs = "PBS_Neurites_vs_Soma"
name_pff = "PFF_Neurites_vs_Soma"

# marker diameters in print, as in the Figure 3C volcano
DOT_BG, DOT_MARKER, DOT_SNCA = native_pt(1.0), native_pt(1.8), native_pt(2.4)

scatter_df = ttest_results[name_pbs][["protein", "log2fc"]].rename(columns={"log2fc": "PBS_log2fc"})
scatter_df = scatter_df.merge(
    ttest_results[name_pff][["protein", "log2fc"]].rename(columns={"log2fc": "PFF_log2fc"}),
    on="protein",
)
scatter_df = scatter_df.merge(adata_log.var[["single_gene_name"]], left_on="protein", right_index=True, how="left")

# Layer of each protein: DCV cargo, DCV machinery, SNCA or background
highlight_markers = {
    "dcv_cargo": dcv_cargo,
    "dcv_machinery": dcv_machinery,
    "snca": ["SNCA"],
}

scatter_df["plot_layer"] = "background"
for layer, genes in highlight_markers.items():
    scatter_df.loc[scatter_df["single_gene_name"].isin(genes), "plot_layer"] = layer

# Colours defined once, so the layers and the legend use the same values.
# Cargo and machinery are two roles in one system but need two hues, not two
# lightnesses: a lighter tint is hard to tell from the grey background points. Teal
# and orange differ on the blue channel, which red-green deficiency preserves —
# under a Vienot simulation the three marks separate at 0.32 / 0.27 / 0.53.
SCATTER_COLORS = {
    "dcv_cargo":     "#44AA99",       # teal
    "dcv_machinery": "#EE7733",       # orange
    "snca":          HIGHLIGHT_COLOR,
    "bg":            BG_COLOR,
}

# The two log2FC columns as an AnnData: layered_plot reads x and y by var name and
# the layer by obs column.
scatter_adata = ad.AnnData(
    X=scatter_df[["PBS_log2fc", "PFF_log2fc"]].values,
    obs=scatter_df[["single_gene_name", "plot_layer"]].reset_index(drop=True),
    var=pd.DataFrame(index=["PBS_log2fc", "PFF_log2fc"]),
)

fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()

# Marker settings per layer; layered_plot passes the "scatter_kwargs" of a layer to scatter().
_marker = {"s": DOT_MARKER ** 2, "alpha": 1.0, "zorder": 5, "edgecolor": "black", "linewidth": MARKER_LW}
at.pl.layered_plot(
    ax=ax,
    base_config=at.pl.make_scatter_config(
        data=scatter_adata,
        x_column="PBS_log2fc",
        y_column="PFF_log2fc",
    ),
    layers=[
        ("plot_layer", "dcv_cargo",     "dcv_cargo",     {"scatter_kwargs": {**_marker}}),
        ("plot_layer", "dcv_machinery", "dcv_machinery", {"scatter_kwargs": {**_marker}}),
        ("plot_layer", "snca",          "snca",          {"scatter_kwargs": {"s": DOT_SNCA ** 2, "alpha": 1.0, "zorder": 6,
                                                                             "marker": "D", "edgecolor": "black",
                                                                             "linewidth": MARKER_LW}}),
        ("plot_layer", "background",    "bg",            {"scatter_kwargs": {"s": DOT_BG ** 2, "alpha": 0.25, "linewidth": 0}}),
    ],
    color_dict=SCATTER_COLORS,
)

# Each axis spans its own data with a 5 % margin: the PBS values (x) reach less
# far into the positive range than the PFF values (y), so a shared range left
# the right part of the panel empty.
def _range(v, margin=0.05):
    lo, hi = np.nanmin(v), np.nanmax(v)
    return lo - margin * (hi - lo), hi + margin * (hi - lo)
xlim = _range(scatter_df["PBS_log2fc"])
ylim = _range(scatter_df["PFF_log2fc"])

# Guide lines: identity (grey) and identity +/- 1 (light grey), drawn across
# both ranges. Red is the UP direction colour, so none of them use it.
lims = [min(xlim[0], ylim[0]), max(xlim[1], ylim[1])]
ax.plot(lims, lims, '--', color=REF_COLOR, linewidth=THIN_LW, zorder=0)
ax.plot(lims, [v + 1 for v in lims], '--', color=BG_COLOR, linewidth=THIN_LW, zorder=0)
ax.plot(lims, [v - 1 for v in lims], '--', color=BG_COLOR, linewidth=THIN_LW, zorder=0)
ax.axhline(0, color=BG_COLOR, linewidth=THIN_LW, zorder=0)
ax.axvline(0, color=BG_COLOR, linewidth=THIN_LW, zorder=0)
ax.set_xlim(xlim); ax.set_ylim(ylim)

# layered_plot writes its own tick sizes from the alphapepttools config, so the
# sizes are set on the axes after plotting.
ax.tick_params(axis="x", labelsize=TICK_FS)
ax.tick_params(axis="y", labelsize=TICK_FS)

# Text labels for SNCA and examples of each group; every DCV protein is coloured.
# Each label is placed at a fixed offset from its dot, in printed points
# (x, y): negative x = left, negative y = down. A grey leader line joins it to
# the dot.
LABEL_OFFSET = {
    "SNCA":   (-8, 4),
    "CHGA":   (0, 6),          # cargo: chromogranin A
    "SCG2":   (-8, 0),         # cargo: secretogranin II
    "PCSK1N": (-8, -4),        # cargo: proSAAS
    "CADPS":  (-2, -8),        # machinery: CAPS1 (DCV priming)
    "RAB3A":  (1, -8),         # machinery
}
labeled = scatter_df[scatter_df["single_gene_name"].isin(LABEL_OFFSET)]
for _, r in labeled.iterrows():
    dx, dy = LABEL_OFFSET[r["single_gene_name"]]
    ax.annotate(r["single_gene_name"], (r["PBS_log2fc"], r["PFF_log2fc"]),
                xytext=(native_pt(dx), native_pt(dy)), textcoords="offset points",
                ha="right" if dx < 0 else ("left" if dx > 0 else "center"),
                va="center" if dy == 0 else ("top" if dy < 0 else "bottom"),
                fontsize=ANNOT_FS, zorder=10,
                bbox=dict(boxstyle="round,pad=0.1", facecolor="white",
                          edgecolor="none", alpha=0.85),
                arrowprops=dict(arrowstyle="-", color=REF_COLOR, lw=THIN_LW,
                                shrinkA=0, shrinkB=DOT_MARKER / 2))

# Legend — colours read from SCATTER_COLORS. SNCA is omitted: its diamond is
# labelled on the panel.
_entry = dict(marker="o", ls="none", ms=DOT_MARKER, mew=MARKER_LW, mec="black")
legend_handles = [
    Line2D([], [], mfc=SCATTER_COLORS["dcv_cargo"],     label="DCV cargo", **_entry),
    Line2D([], [], mfc=SCATTER_COLORS["dcv_machinery"], label="DCV machinery", **_entry),
]
# handlelength / handletextpad: the marker slot and its gap to the text (defaults
# 2.0 and 0.8, in font sizes); borderaxespad: distance to the axes corner (default 0.5).
ax.legend(handles=legend_handles, loc="lower right", frameon=False,
          handlelength=1.0, handletextpad=0.3, borderaxespad=0,
          bbox_to_anchor=(1.05, 0))

label_axes(ax, xlabel="PBS log2FC (Neurites / Soma)",
               ylabel="PFF log2FC (Neurites / Soma)")

save_panel(fig, "e", "scatter_PBS_vs_PFF_dcv_markers", FIG3_DIR)

## 10.2 Dense-core vesicle proteins across comparisons (SI4E)

In [ ]:
# SI4E, 1 x 1 unit: log2FC of the DCV cargo and machinery in four comparisons
# (t-tests of section 7). Drawn by group_heatmap (src/heatmap.py), which lists
# the DCV proteins not in the strict t-test tables. SNCA is not shown here.
# Own names (dcv_*), so a re-run of the Figure 3D cell after this one still reads
# its own comparisons.
dcv_comparisons = [
    "PBS_Neurites_vs_Soma",
    "PFF_Neurites_vs_Soma",
    "Neuritic_Agg_vs_Neurites",
    "Somatic_Agg_vs_Soma",
]
# Short column names of the GSEA figure, horizontal on two lines: rotated by
# 90 degrees they took about 10 mm of the panel height.
dcv_labels = ["PBS\nN/S", "PFF\nN/S", "NAgg/\nNeur", "SAgg/\nSoma"]

# log2FC matrix: gene symbols x comparisons
heatmap_data = {}
for comp in dcv_comparisons:
    result = ttest_results[comp]
    result = result.merge(adata_log.var[["single_gene_name"]],
                          left_on="protein", right_index=True, how="left")
    heatmap_data[comp] = result.set_index("single_gene_name")["log2fc"]
lfc_matrix = pd.DataFrame(heatmap_data)
lfc_matrix.columns = dcv_labels

# Gene names print at 5.5 pt, 0.5 pt below the style size: the rows of the
# 1 x 1 panel are 4.6 pt apart.
group_heatmap(lfc_matrix, [("Cargo", dcv_cargo), ("Machinery", dcv_machinery)],
              "e", "heatmap_dcv_markers", SI4_DIR, size=(1, 1), gene_fs=native_pt(5.5),
              label_gap_pt=1.5, cbar_shrink=0.6)

# 11. Organelle markers (SI4F)

In [ ]:
# ============================================================
# Organelle marker score per sample and family (SI4F, plotted in the next cell)
# ============================================================
# Each marker protein is centred on its own mean over all samples (log2
# intensity minus the protein mean), and the centred values of a family's
# markers are averaged per sample. Centring puts the four families on one scale
# around 0. Without it a sample's mean would also depend on the absolute
# abundance of the markers it has values for, and the families would sit at
# different heights of the shared axis. Input: adata_log (complete in at least one
# condition, log2, not imputed). Curated markers (gene: description; the
# description is for the reader and is not drawn).
mito_markers = {
    "NDUFA9":  "NDUFA9 (Complex I)",
    "TOMM20":  "TOMM20 (outer-membrane translocase)",
    "CYCS":    "Cytochrome c",
    "COX4I1":  "COX4I1 (Complex IV)",
    "VDAC1":   "VDAC1 (outer membrane)",
    "HSPD1":   "HSP60 (mitochondrial chaperonin)",
    "ATP5F1B": "ATP5F1B (Complex V)",
}
lyso_markers = {
    "CTSB":   "Cathepsin B",
    "GBA1":   "β-glucocerebrosidase (GBA1)",
    "LAMP1":  "LAMP1",
    "LAMP2":  "LAMP2",
    "SCARB2": "LIMP2 (SCARB2)",
    "CTSD":   "Cathepsin D",
}
er_markers = {
    "P4HB":  "P4HB / PDI",
    "CALR":  "Calreticulin",
    "CANX":  "Calnexin",
    "RTN4":  "Reticulon-4 (ER shaper)",
    "HSPA5": "BiP (HSPA5)",
}
golgi_markers = {
    "GOLGA2": "GM130 (GOLGA2)",
    "GOLGA3": "Golgin A3",
    "COPA":   "α-COP",
    "COPB1":  "β-COP",
    "RAB6A":  "RAB6A (Golgi GTPase)",
}

all_organelle_markers = {
    "mitochondrial": mito_markers,
    "lysosomal":     lyso_markers,
    "ER":            er_markers,
    "Golgi":         golgi_markers,
}

# Per family: the samples x markers matrix, each marker centred on its own mean,
# then the mean over the markers per sample. A marker missing from the data is
# reported and left out.
conds = adata_log.obs[biological].astype(str).values
keep = np.isin(conds, biological_order)

family_genes = {}   # family -> markers found in the data
rows = []
for family_name, marker_dict in all_organelle_markers.items():
    genes, protein_ids = [], []
    for gene in marker_dict:
        match = adata_log.var[adata_log.var["single_gene_name"] == gene]
        if match.empty:
            print(f"{gene} not found in dataset (family: {family_name})")
            continue
        genes.append(gene)
        protein_ids.append(match.index[0])

    if not genes:
        print(f"Skipping {family_name} — no genes detected")
        continue

    vals = np.asarray(adata_log[keep, protein_ids].X, dtype=float)
    n_missing = int(np.isnan(vals).sum())
    if n_missing:
        print(f"CHECK {family_name}: {n_missing} of {vals.size} marker values are missing; "
              "those samples are averaged over the markers they have values for")
    centred = vals - np.nanmean(vals, axis=0)
    score = np.nanmean(centred, axis=1)

    family_genes[family_name] = genes
    rows += [{"Family": family_name, "Condition": c, "Score": s}
             for c, s in zip(conds[keep], score)]

df_plot = pd.DataFrame(rows)
families = list(family_genes)

In [ ]:
# SI4F, 4 x 1 units: organelle marker score per sample (previous cell), one box
# per condition and family. Colour = aggregate status (grey PBS, blue
# aggregate-free, violet excised aggregate), CATEGORY_COLORS of the style cell.
# The compartment is the hatch: at the box alpha used here the two lightnesses of
# one hue could not be told apart; hatching also survives greyscale printing.

# Family names, written out: str.capitalize() lowercases everything after the
# first letter, so "ER" would print as "Er".
FAMILY_TITLE = {
    "mitochondrial": "Mitochondrial markers",
    "lysosomal":     "Lysosomal markers",
    "ER":            "ER markers",
    "Golgi":         "Golgi markers",
}

# Neurites block first (PBS -> PFF -> Neuritic Agg), then Soma block
# (PBS -> PFF -> Somatic Agg). This is the panel's layout order and is
# deliberately not biological_order; the colours do not depend on it.
condition_order = [
    "iDA D56 PBS N1 Neurites Without Aggregates",
    "iDA D56 PFF N2 Neurites Without Aggregates",
    "iDA D56 PFF N2 Aggregates Neuritic",
    "iDA D56 PBS N1 Soma Without Aggregates",
    "iDA D56 PFF N2 Soma Without Aggregates",
    "iDA D56 PFF N2 Aggregates Somatic",
]
assert set(condition_order) == set(CATEGORY_COLORS), (
    "condition_order and CATEGORY_COLORS disagree on the category set: "
    f"{sorted(set(condition_order) ^ set(CATEGORY_COLORS))}"
)
condition_colors = {c: CATEGORY_COLORS[c] for c in condition_order}
# Neurites hatched, soma solid.
condition_hatch = {c: ("///" if "Neurites" in c or "Neuritic" in c else "") for c in condition_order}

# --- Layout parameters ---
# Families are one unit apart. GROUP_W is how much of that unit the six boxes and
# the compartment gap occupy together, so raising it widens every box and the
# remainder is the gap to the next family.
GROUP_W   = 0.80   # fraction of the family pitch covered by the six boxes
COMP_GAP  = 0.06   # gap between the Neurites block and the Soma block
DOT       = native_pt(1.6)   # jitter dot diameter, 1.6 pt in print
BOX_ALPHA = 0.80

BOX_W = (GROUP_W - COMP_GAP) / len(condition_order)

# Box offsets within one family, centred on the family position
box_offsets = []
for k in range(len(condition_order)):
    base = k * BOX_W
    if k >= 3:
        base += COMP_GAP
    box_offsets.append(base)
total_family_w = box_offsets[-1] + BOX_W
box_offsets = [b - total_family_w / 2 + BOX_W / 2 for b in box_offsets]

x = np.arange(len(families), dtype=float)

fig, axm = create_figure(1, 1, figsize=figsize(4, 1))
ax = axm.next()

# One random generator for the panel, so the families get different jitter.
rng = np.random.default_rng(42)

for k, cond in enumerate(condition_order):
    cond_data = df_plot[df_plot["Condition"] == cond]
    positions = x + box_offsets[k]
    bp_data = [cond_data[cond_data["Family"] == f]["Score"].dropna().values for f in families]

    bp = ax.boxplot(bp_data, positions=positions, widths=BOX_W * 0.92,
                    patch_artist=True, showfliers=False, zorder=2)
    color = condition_colors[cond]
    # Box, whisker, cap and median widths come from the style's boxplot settings.
    for patch in bp["boxes"]:
        patch.set_facecolor(color)
        patch.set_alpha(BOX_ALPHA)
        patch.set_edgecolor("black")
        patch.set_hatch(condition_hatch[cond])

    for j in range(len(families)):
        vals = bp_data[j]
        jitter = rng.uniform(-BOX_W * 0.18, BOX_W * 0.18, size=len(vals))
        ax.scatter(positions[j] + jitter, vals, color=color,
                   s=DOT ** 2, alpha=1.0, edgecolor="black", linewidth=MARKER_LW, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels([f"{FAMILY_TITLE[f]}\n(mean of {len(family_genes[f])} proteins)" for f in families])
ax.set_xlim(-0.5 - GROUP_W / 4, (len(families) - 1) + 0.5 + GROUP_W / 4)
label_axes(ax, ylabel="log2 intensity\n(centred per protein)")

# One legend row under the family names, in the left-to-right order of the boxes.
cond_handles = [
    Patch(facecolor=condition_colors[c], alpha=BOX_ALPHA, edgecolor="black",
          hatch=condition_hatch[c], label=CATEGORY_SHORT[c])
    for c in condition_order
]
fig.legend(handles=cond_handles, loc="outside lower center", ncol=len(cond_handles),
           frameon=False, borderaxespad=0, borderpad=0.2,
           handlelength=1.5, columnspacing=1.0)
fig.get_layout_engine().set(h_pad=0.01)

save_panel(fig, "f", "organelle_mean", SI4_DIR)

# 12. Aggregate composition (Figure 3F)

In [ ]:
# ============================================================
# Figure 3F rows, selected from the data: the top TOP_N_UP enriched and the top
# TOP_N_DOWN depleted proteins (FDR < 0.05) of each aggregate contrast (t-tests
# of section 7), combined without duplicates. The next cell groups the same genes
# by function; the heatmap cell asserts that both hold the same genes.
# ============================================================
TOP_N_UP   = 15   # top enriched (log2fc > 0) per comparison
TOP_N_DOWN = 5    # top depleted (log2fc < 0) per comparison

# Comparisons in order. Own names (agg_*), so a re-run of the Figure 3D cell after
# this section still reads its own comparisons.
agg_comparisons = [
    "Neuritic_Agg_vs_Neurites",
    "Somatic_Agg_vs_Soma",
    "Neuritic_Agg_vs_Soma_Agg",
]
agg_labels = [
    "Neuritic Agg / Neurites",
    "Somatic Agg / Soma",
    "Neuritic Agg / Somatic Agg",
]

# ttest_results were computed on adata_to_use (t-test cell), so the gene names
# are read from the same object.
name_map = adata_to_use.var["single_gene_name"].to_dict()

# Top-UP and top-DOWN proteins per comparison
topN_up = {}
topN_down = {}
for comp, label in zip(agg_comparisons, agg_labels):
    res = ttest_results[comp].copy()
    res = res.merge(adata_to_use.var[["single_gene_name"]],
                    left_on="protein", right_index=True, how="left")
    sig = res[res["fdr"] < 0.05]

    up   = sig[sig["log2fc"] > 0].nlargest(TOP_N_UP, "log2fc")
    down = sig[sig["log2fc"] < 0].nsmallest(TOP_N_DOWN, "log2fc")
    topN_up[label]   = set(up["protein"].tolist())
    topN_down[label] = set(down["protein"].tolist())

    print(f"\n========== {label} ==========")
    print(f"  Top {TOP_N_UP} UP    ({len(up)}):")
    for _, row in up.iterrows():
        print(f"    {str(row['single_gene_name']):15s}  log2fc={row['log2fc']:+.2f}  fdr={row['fdr']:.2e}")
    print(f"  Top {TOP_N_DOWN} DOWN  ({len(down)}):")
    for _, row in down.iterrows():
        print(f"    {str(row['single_gene_name']):15s}  log2fc={row['log2fc']:+.2f}  fdr={row['fdr']:.2e}")

# The rows of Figure 3F: the union of all top-UP and top-DOWN proteins
combined_set = set().union(*topN_up.values()) | set().union(*topN_down.values())
combined = sorted(combined_set, key=lambda p: str(name_map.get(p, p)))

print(f"\nCombined unique: {len(combined)}")
for i, p in enumerate(combined, 1):
    print(f"  {i:2d}. {str(name_map.get(p, p)):15s}  [{p}]")

In [ ]:
# The Figure 3F rows of the previous cell, grouped by function by hand (gene:
# description; the description is for the reader and is not drawn).

# --- ENRICHED IN iPSC AGGREGATES ---

snca = {
    "SNCA": "α-synuclein",
}

aggrephagy = {
    "SQSTM1": "p62 / Sequestosome-1",
    "OPTN":   "Optineurin",
    "ATG9A":  "ATG9A",
}

proteostasis = {
    "UBQLN2":  "Ubiquilin-2",
    "MLF2":    "Myeloid leukemia factor 2",
    "BAG6":    "BAG6 (BAT3 complex)",
    "UBL4A":   "UBL4A (BAT3 complex)",
    "UBB":     "Ubiquitin (UBB)",
    "UFD1":    "UFD1 (VCP cofactor)",
    "CHMP2A":  "CHMP2A (ESCRT-III)",
    "ZC2HC1A": "ZC2HC1A",
}

if_cage = {
    "NES":  "Nestin",
    "INA":  "α-internexin",
    "PRPH": "Peripherin",
    "NEFM": "Neurofilament medium",
}

microtubule_scaffold = {
    "MAP4":    "MAP4",
    "CLIP1":   "CLIP1 (+TIP)",
    "EPB41L3": "Band 4.1-like 3 (sub-membrane scaffold)",
    "ANK2":    "Ankyrin-B",
}

axon_signaling = {
    "PLXNA4": "Plexin-A4 (axon guidance)",
    "NRAS":   "NRAS (small GTPase)",
    "GNB1":   "G-protein β1",
}

synaptic_vesicle = {
    "SYT11": "Synaptotagmin-11",
    "EPS15": "EPS15 (clathrin adaptor)",
    "AP3B2": "AP3B2 (neuronal AP-3)",
    "PACS2": "PACS2",
    "ITM2B": "ITM2B / BRI2",
}

transporters = {
    "SLC2A3":   "GLUT3",
    "SLC18A3":  "VAChT (SLC18A3)",
    "SLC9A6":   "Na⁺/H⁺ exchanger 6",
    "SLC39A10": "ZIP10 (Zn²⁺ transporter)",
    "ATP9A":    "ATP9A (P4-ATPase)",
    "CLCN3":    "CLCN3 (Cl⁻/H⁺ exchanger)",
    "TRPV2":    "TRPV2 (Ca²⁺ channel)",
}

ecm_basement = {
    "LAMA1":  "Laminin α1",
    "LAMB1":  "Laminin β1",
    "LAMC1":  "Laminin γ1",
    "NID1":   "Nidogen-1",
    "B3GAT3": "B3GAT3 (GAG biosynthesis)",
}


# --- DEPLETED IN iPSC AGGREGATES ---

nuclear_depleted = {
    "EWSR1":     "EWS",
    "MACROH2A2": "macroH2A2",
    "YBX1":      "YB-1",
    "NCOA5":     "NCOA5",
    "API5":      "API5",
}

dendritic_depleted = {
    "MAP2": "MAP2 (dendritic)",
}

other_depleted = {
    "PSAP":     "Saposin (PSAP)",
    "LGALS3BP": "LGALS3BP / Mac-2BP",
    "CPVL":     "Carboxypeptidase VL",
    "PRKRA":    "PRKRA",
    "GRAMD1A":  "GRAMD1A",
    "EEA1":     "Early endosome antigen 1",
    "FARSB":    "FARSB (Phe-tRNA synthetase)",
    "CMAS":     "CMAS (sialic acid synthesis)",
    "HINT3":    "HINT3",
}


# Heatmap groups in row order: (label, marker dict). One line per label: a
# two-line label doubles the width of the label column, and the heatmap columns
# need that width. SNCA is the first row, without a label.
enriched_groups = [
    ("",             snca),
    # The aggrephagy receptors (SQSTM1, OPTN, ATG9A) are the first rows of
    # "Proteostasis": their own 3-row block could not hold the rotated label.
    ("Proteostasis", {**aggrephagy, **proteostasis}),
    ("IF cage",      if_cage),
    ("Scaffold",     microtubule_scaffold),   # cytoskeleton: MAP4, CLIP1, EPB41L3, ANK2
    ("Signalling",   axon_signaling),         # axon guidance: PLXNA4, NRAS, GNB1
    ("Vesicles",     synaptic_vesicle),       # vesicle trafficking
    ("Transporters", transporters),
    ("ECM",          ecm_basement),
]

depleted_groups = [
    ("Nuclear", nuclear_depleted),
    # MAP2 is in "Other": a one-row block cannot hold its rotated label, and the
    # row itself names MAP2.
    ("Other",   {**dendritic_depleted, **other_depleted}),
]
agg_groups = enriched_groups + depleted_groups

# Heatmap rows, top to bottom. Own names (agg_*), as in the cell above.
agg_row_order = [g for _, markers in agg_groups for g in markers]

In [ ]:
# ============================================================
# Figure 3F, 1 x 2 units: log2FC of the aggregate composition rows in the three
# aggregate contrasts (t-tests of section 7), grouped by function, with a thicker
# line between the enriched and the depleted groups. Drawn by group_heatmap
# (src/heatmap.py); the colour scale is symmetric at the largest |log2FC| drawn.
# ============================================================
AGG_SIZE = (1, 2)

# Gene names print at 4.5 pt (5.0 x 0.9), below Nature's 5 pt minimum, set on
# purpose: at 1 x 2 units the 55 rows are 4.0 pt apart.
GENE_FS = native_pt(5.0)*0.9

# Column names horizontal on two lines, numerator over denominator. Rotated, they
# took 6.5 mm of the height; horizontal, 4.0 mm.
short_agg_labels = ["NAgg/\nNeur", "SAgg/\nSoma", "NAgg/\nSAgg"]

# The grouped rows are exactly the top-N selection of the first cell of this
# section, so the figure follows the stated selection rule.
_selected = {name_map[p] for p in combined}
assert set(agg_row_order) == _selected, (
    f"grouped rows not selected: {sorted(set(agg_row_order) - _selected)}; "
    f"selected, not grouped: {sorted(_selected - set(agg_row_order))}")

# single_gene_name is not unique across protein groups. Pivoting on it keeps
# whichever row pandas saw last, without a warning; the representative is chosen
# by abundance.
_med = pd.Series(np.nanmedian(adata_to_use.X, axis=0), index=adata_to_use.var_names)
_map = (adata_to_use.var[["single_gene_name"]].assign(_abund=_med)
        .dropna(subset=["single_gene_name"])
        .sort_values("_abund", ascending=False)
        .drop_duplicates("single_gene_name"))
print(f"Gene symbols: {len(_map)} representatives from "
      f"{adata_to_use.n_vars} protein groups "
      f"({adata_to_use.n_vars - len(_map)} collapsed onto a taken symbol)")

# log2FC matrix: gene symbols x comparisons
heatmap_data = {}
for comp in agg_comparisons:
    res = ttest_results[comp].merge(_map[["single_gene_name"]],
                                    left_on="protein", right_index=True, how="inner")
    heatmap_data[comp] = res.set_index("single_gene_name")["log2fc"]
lfc_matrix = pd.DataFrame(heatmap_data)
lfc_matrix.columns = short_agg_labels

# Group names at two depths, so that a name longer than its block keeps its full
# length.
group_heatmap(lfc_matrix, agg_groups, "f", "heatmap_aggregate_composition", FIG3_DIR,
              size=AGG_SIZE, gene_fs=GENE_FS, stagger=2, block_after=len(enriched_groups),
              col_linespacing=1.0)

# Quoted in the text: UBQLN2 in the neuritic aggregates, SNCA in both aggregates.
for comp, gene in (("Neuritic_Agg_vs_Neurites", "UBQLN2"),
                   ("Neuritic_Agg_vs_Neurites", "SNCA"),
                   ("Somatic_Agg_vs_Soma", "SNCA")):
    _v = heatmap_data[comp][gene]
    print(f"CHECK {gene} {comp}: log2FC {_v:+.2f}, {2 ** _v:.1f}-fold")

# 13. Ubiquitin-dependent quality control, targeted panel (Figure 3G)

In [ ]:
# ============================================================
# Targeted panel [1/3]: ubiquitin-dependent quality-control proteins, tested per
# contrast (Figure 3G). Writes the long table tp_long as a CSV.
# ============================================================
# The t-tests in section 7 run on adata_strict, which keeps only proteins with a
# value in all 33 samples. A protein missing from one PBS soma sample is then
# also dropped from the aggregate contrasts, which do not contain PBS soma.
# Here each contrast is filtered on its own two groups: a protein is tested when
# it has a value in at least TP_MIN_FRAC of the samples of each group (and in at
# least 2). Values are the measured log2 intensities of adata_sf, without
# imputation. The t-test runs on every protein that passes the filter, so "fdr"
# is the proteome-wide Benjamini-Hochberg FDR of the contrast; "fdr_panel" is the
# Benjamini-Hochberg FDR within the panel proteins tested in that contrast.

TP_MIN_FRAC = 0.5   # 3 of 6 samples; 2 of 3 for the somatic aggregates

# label: (group1, group2); log2fc = group1 minus group2
TP_CONTRASTS = {
    "PFF Neurites / PBS Neurites": ("iDA D56 PFF N2 Neurites Without Aggregates",
                                    "iDA D56 PBS N1 Neurites Without Aggregates"),
    "PFF Soma / PBS Soma":         ("iDA D56 PFF N2 Soma Without Aggregates",
                                    "iDA D56 PBS N1 Soma Without Aggregates"),
    "Neuritic Agg / PFF Neurites": ("iDA D56 PFF N2 Aggregates Neuritic",
                                    "iDA D56 PFF N2 Neurites Without Aggregates"),
    "Somatic Agg / PFF Soma":      ("iDA D56 PFF N2 Aggregates Somatic",
                                    "iDA D56 PFF N2 Soma Without Aggregates"),
    "Neuritic Agg / Somatic Agg":  ("iDA D56 PFF N2 Aggregates Neuritic",
                                    "iDA D56 PFF N2 Aggregates Somatic"),
}

TP_FAMILIES = {
    "Ubiquilins":              ["UBQLN1", "UBQLN2", "UBQLN4"],
    "19S ubiquitin receptors": ["PSMD4", "ADRM1"],
    "p97":                     ["VCP", "UFD1", "NPLOC4", "NSFL1C", "UBXN1", "UBXN4", "UBXN6", "FAF2"],
    "BAG6 axis":               ["BAG6", "UBL4A", "GET4", "GET3", "SGTA", "RNF126"],
    "ERAD":                    ["SYVN1", "SEL1L", "DERL1", "UBE2J1", "OS9"],
    "RQC":                     ["NEMF", "LTN1", "ANKZF1", "ZNF598", "RACK1", "ABCE1"],
    "Other shuttle":           ["RAD23B", "DDI2"],
    "Reference":               ["SNCA", "SQSTM1", "OPTN", "MLF2"],
}
# Proteasome subunits by gene-name pattern. A protein that is already in a named
# family above stays there, so PSMD4 is reported once, as a 19S ubiquitin receptor.
TP_REGEX_FAMILIES = {
    "Proteasome PSMC": r"PSMC\d+",
    "Proteasome PSMD": r"PSMD\d+",
    "Proteasome PSMA": r"PSMA\d+",
    "Proteasome PSMB": r"PSMB\d+",
}
TP_FAMILY_ORDER = list(TP_FAMILIES) + list(TP_REGEX_FAMILIES)

# --- Matrix: all proteins of the sample-filtered data, log2, contaminants removed
# (flags of the preprocessing cell)
tp_adata = adata_sf.copy()
sc.pp.log1p(tp_adata, base=2)
tp_adata = tp_adata[:, ~tp_adata.var["is_contaminant"]].copy()
print(f"Matrix: {tp_adata.n_obs} samples, {tp_adata.n_vars} proteins "
      f"(adata_sf without contaminants, log2, not imputed)")

# --- Panel: match every gene of a protein group, named families first
_named = {g: fam for fam, genes in TP_FAMILIES.items() for g in genes}
_rows = []
for pg, gene_list in tp_adata.var["genes"].astype(str).str.split(";").items():
    fam = next((_named[g] for g in gene_list if g in _named), None)
    if fam is None:
        fam = next((f for f, pat in TP_REGEX_FAMILIES.items()
                    if any(re.fullmatch(pat, g) for g in gene_list)), None)
    if fam is not None:
        _rows.append({"pg": pg, "family": fam, "gene": tp_adata.var.loc[pg, "single_gene_name"]})
tp_panel = pd.DataFrame(_rows).set_index("pg")

# Order: family order above, list order within named families, number order
# within the proteasome families (PSMD2 before PSMD10).
_list_pos = {g: i for genes in TP_FAMILIES.values() for i, g in enumerate(genes)}
tp_panel["_f"] = tp_panel["family"].map(TP_FAMILY_ORDER.index)
tp_panel["_g"] = tp_panel["gene"].map(
    lambda g: _list_pos.get(g, int(re.sub(r"\D", "", g) or 0)))
tp_panel = tp_panel.sort_values(["_f", "_g"]).drop(columns=["_f", "_g"])
tp_panel["in_strict"] = tp_panel.index.isin(adata_strict_no_contaminants.var_names)

_found = {g for pg in tp_panel.index for g in str(tp_adata.var.loc[pg, "genes"]).split(";")}
_absent = [g for genes in TP_FAMILIES.values() for g in genes if g not in _found]
print(f"Panel: {len(tp_panel)} protein groups; named genes without any value: {_absent or 'none'}")
print(f"In the strict t-test input (section 7): {tp_panel['in_strict'].sum()} of {len(tp_panel)}")

# --- Detection counts per condition (values present, of samples in the condition)
tp_n = tp_adata.obs[biological].value_counts()
tp_counts = pd.DataFrame(
    {c: (~np.isnan(tp_adata[tp_adata.obs[biological] == c].X)).sum(axis=0) for c in biological_order},
    index=tp_adata.var_names,
)

# --- t-tests with the per-contrast filter
_parts = []
for label, (g1, g2) in TP_CONTRASTS.items():
    k1 = max(2, int(np.ceil(TP_MIN_FRAC * tp_n[g1])))
    k2 = max(2, int(np.ceil(TP_MIN_FRAC * tp_n[g2])))
    keep = (tp_counts[g1] >= k1) & (tp_counts[g2] >= k2)
    res = at.tl.diff_exp_ttest(
        adata=tp_adata[:, keep.values].copy(),
        between_column=biological,
        comparison=(g1, g2),
    ).set_index("protein")

    part = tp_panel.copy()
    part.insert(0, "contrast", label)
    part["n1"], part["N1"] = tp_counts.loc[part.index, g1].values, tp_n[g1]
    part["n2"], part["N2"] = tp_counts.loc[part.index, g2].values, tp_n[g2]
    part["tested"] = keep.loc[part.index].values
    part = part.join(res[["log2fc", "p_value", "fdr"]])
    part["fdr_panel"] = at.tl.nan_safe_bh_correction(part["p_value"].to_numpy())
    _parts.append(part)
    print(f"{label:28s} rule >= {k1}/{tp_n[g1]} and >= {k2}/{tp_n[g2]}: "
          f"{keep.sum()} proteins tested; panel {part['tested'].sum()} of {len(part)} "
          f"(strict filter: {part['in_strict'].sum()})")

tp_long = pd.concat(_parts).rename_axis("protein").reset_index()
tp_long = tp_long.merge(tp_adata.var[["proteins", "uniprot_ids"]],
                        left_on="protein", right_index=True, how="left")

_path = f"{output_dir}/{today}_targeted_panel_per_contrast_min{int(TP_MIN_FRAC * 100)}pct.csv"
tp_long.to_csv(_path, index=False)
print(f"\nWritten: {_path}")

In [ ]:
# ============================================================
# Targeted panel [2/3]: summary tables of tp_long (cell above). The per-family
# table is the source of the family-level statements on Figure 3G in the text;
# the detection differences are the proteins the t-test cannot assess.
# ============================================================
# FDR_COL picks the correction behind the stars, the up/down counts and the dot
# plot (next cell). The panel was assembled after UBQLN2 led the neuritic
# contrast, so the correction is the proteome-wide one of the contrast ("fdr");
# "fdr_panel", within the panel only, stays in the table.
FDR_COL = "fdr"
FDR_CUT = 0.05


def _stars(q):
    return "***" if q < 0.001 else "**" if q < 0.01 else "*" if q < FDR_CUT else ""


_labels = list(TP_CONTRASTS)
tp_long["sig"] = tp_long[FDR_COL] < FDR_CUT

# --- 1. Per family: up and down at the FDR cut-off, of tested; median log2FC of tested
def _family_cell(d):
    t = d[d["tested"]]
    if t.empty:
        return "not tested"
    up   = (t["sig"] & (t["log2fc"] > 0)).sum()
    down = (t["sig"] & (t["log2fc"] < 0)).sum()
    return f"{up} up, {down} down of {len(t)}; median {t['log2fc'].median():+.2f}"

tp_family_summary = pd.DataFrame({
    label: {fam: _family_cell(tp_long[(tp_long["contrast"] == label) & (tp_long["family"] == fam)])
            for fam in TP_FAMILY_ORDER}
    for label in _labels
})
print(f"Per family ({FDR_COL} < {FDR_CUT}; log2FC = first group minus second group)")
display(tp_family_summary)

# --- 2. Per protein: log2FC with stars if tested; detection counts if not tested
def _protein_cell(r):
    if r["tested"]:
        return f"{r['log2fc']:+.2f}{_stars(r[FDR_COL])}"
    return f"n.t. {r['n1']}/{r['N1']} vs {r['n2']}/{r['N2']}"

tp_long["cell"] = tp_long.apply(_protein_cell, axis=1)
tp_protein_summary = (
    tp_long.pivot(index="protein", columns="contrast", values="cell")
    .reindex(index=tp_panel.index, columns=_labels)
)
tp_protein_summary.index = pd.MultiIndex.from_frame(
    tp_panel[["family", "gene"]].assign(strict=np.where(tp_panel["in_strict"], "", "not in strict")))
print(f"\nPer protein: log2FC, * {FDR_COL} < 0.05, ** < 0.01, *** < 0.001; "
      f"n.t. = not tested, values present in group 1 vs group 2")
with pd.option_context("display.max_rows", 200, "display.max_columns", 20):
    display(tp_protein_summary)

# --- 3. Detection differences: not tested because one group has too few values
# while the other group passes. The t-test cannot assess these; the counts are
# the result. A low count can mean below the detection limit or absent.
_rows = []
for label, (g1, g2) in TP_CONTRASTS.items():
    k1 = max(2, int(np.ceil(TP_MIN_FRAC * tp_n[g1])))
    k2 = max(2, int(np.ceil(TP_MIN_FRAC * tp_n[g2])))
    d = tp_long[(tp_long["contrast"] == label) & ~tp_long["tested"]]
    for _, r in d.iterrows():
        if (r["n1"] >= k1) != (r["n2"] >= k2):
            higher = CATEGORY_SHORT[g1 if r["n1"] >= k1 else g2]
            _rows.append({"contrast": label, "family": r["family"], "gene": r["gene"],
                          "group 1": f"{r['n1']}/{r['N1']}", "group 2": f"{r['n2']}/{r['N2']}",
                          "detected in": higher})
print("\nDetection differences (not tested; one group passes the filter, the other does not)")
display(pd.DataFrame(_rows) if _rows else "none")

In [ ]:
# ============================================================
# Targeted panel [3/3]: Figure 3G, 1 x 2 units. Dot plot of protein quality
# control and proteasome subunits against the five contrasts, drawn by
# tested_dotplot (src/dotplot.py) from tp_long.
# ============================================================
# The encoding of the BafA1 dot plot (Autophagy Figure 2G): colour = log2FC,
# dot area = -log10 FDR (FDR_COL of the summary cell), black outline = FDR < 0.05,
# grey cross = not tested (fewer than TP_MIN_FRAC of the samples of a group have
# a value).

# Column labels: the short contrast names of the GSEA figure, rotated. The full
# names took 28 mm of the panel height, these take 12 mm.
TP_SHORT_LABELS = {
    "PFF Neurites / PBS Neurites": "Neur PFF/PBS",
    "PFF Soma / PBS Soma":         "Soma PFF/PBS",
    "Neuritic Agg / PFF Neurites": "NAgg/Neur",
    "Somatic Agg / PFF Soma":      "SAgg/Soma",
    "Neuritic Agg / Somatic Agg":  "NAgg/SAgg",
}

# Groups and rows, top to bottom, in this order. A label longer than its group
# runs past the group's bracket; the dot plot reports neighbouring labels that
# overlap. "\n" in a group name starts a second line. The 19S and 20S groups hold
# the two ubiquitin receptors and three subunits per subcomplex, listed by hand.
TP_MAIN_GROUPS = [
    ("Shuttles",     ["UBQLN2", "UBQLN1", "UBQLN4", "RAD23B"]),
    ("BAG6 / p97",   ["RNF126", "UBL4A", "BAG6", "GET4", "SGTA",
                      "UBXN1", "UFD1", "UBXN6", "NPLOC4", "NSFL1C", "VCP"]),
    ("ERAD /\nRQC",  ["SEL1L", "DERL1", "NEMF", "RACK1"]),
    ("19S",          ["ADRM1", "PSMD4",                     # ubiquitin receptors
                      "PSMC5", "PSMC2", "PSMC3", "PSMD3", "PSMD7", "PSMD2"]),
    ("20S",          ["PSMA5", "PSMA2", "PSMA4", "PSMB1", "PSMB4", "PSMB5"]),
    ("Other",         ["SNCA", "SQSTM1", "OPTN", "MLF2"]),     # reference proteins
]
TP_MAIN_SIZE = (1, 2)    # units
_shown = [g for _, genes in TP_MAIN_GROUPS for g in genes]
assert len(_shown) == len(set(_shown)), "a gene is in two groups"
print(f"{len(_shown)} rows in {len(TP_MAIN_GROUPS)} groups")

# The colour scale ends at +-TP_VMAX: SNCA (+7.9 in Neur PFF/PBS) would
# otherwise leave every other dot near white, and the proteasome changes are
# about 0.5. Values beyond it take the end colour.
TP_VMAX = 1.5

# Dot area is linear in -log10 FDR up to FDR = 10^-TP_SIZE_CAP, constant beyond.
TP_SIZE_CAP = 4

tested_dotplot(TP_MAIN_GROUPS, tp_long, list(TP_CONTRASTS),
               [TP_SHORT_LABELS[c] for c in TP_CONTRASTS],
               "g", "targeted_panel_dotplot", FIG3_DIR, size=TP_MAIN_SIZE,
               fdr_col=FDR_COL, vmax=TP_VMAX, size_cap=TP_SIZE_CAP)

# 14. Figure assembly

## 14.1 Figure 3

In [ ]:
# Figure 3 from the saved panels. Layout: one letter per grid unit, "." = empty.
# Panel h is the immunofluorescence image, an SVG made outside this notebook and
# not part of the repository; panel_scale exempts it from the whole-unit width
# check. compose_figure looks for a file h_*.svg in FIG3_DIR and stops with
# FileNotFoundError when there is none. Put the image there, or replace "h" by "."
# in the layout and delete "h" from panel_scale to assemble the other panels.
figure_name = f"{today}_Figure3_ipscs"
layout = "a b c e\n d f g .\n d f g ."

compose_figure(
    folder=FIG3_DIR,
    layout=layout,
    output=f"{FIG3_DIR}/{figure_name}.svg",
    show_box=True,
    total_width_mm=180,
    panel_offset_mm={"e": (1, 1)},
    # panel_scale={"h": 1.0},
)

## 14.2 SI4

In [ ]:
# SI4 from the saved panels. Layout: one letter per grid unit.
figure_name = f"{today}_SI4_IPSCs"
layout = "a a b e\nc c d d\nc c d d\n f f f f"
compose_figure(
    folder=SI4_DIR,
    layout=layout,
    output=f"{SI4_DIR}/{figure_name}.svg",
    show_box=True,
)